In [1]:
!pip install -q yfinance xgboost lightgbm scikit-learn

In [8]:
!pip install -q yfinance xgboost scikit-learn

In [9]:
from google.colab import drive
drive.mount("/content/drive")

import pandas as pd
import numpy as np
import re
import yfinance as yf
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

DATA_DIR = Path("/content/drive/MyDrive/bigquery")

NEWS_TOPIC_FILE = DATA_DIR / "news_clean_topic_count_only.csv"

EVENT_START = "2026-02-15"
EVENT_END = "2026-04-27"
FREQ = "4h"

print("News topic file exists:", NEWS_TOPIC_FILE.exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
News topic file exists: True


In [10]:
news = pd.read_csv(NEWS_TOPIC_FILE, low_memory=False)

news["published_utc"] = pd.to_datetime(news["published_utc"], errors="coerce", utc=True)
news = news.dropna(subset=["published_utc", "title_clean", "topic"]).copy()

news = news[
    (news["published_utc"] >= pd.Timestamp(EVENT_START, tz="UTC")) &
    (news["published_utc"] < pd.Timestamp(EVENT_END, tz="UTC"))
].copy()

if "relevance_score" not in news.columns:
    news["relevance_score"] = 1

if "source" not in news.columns:
    news["source"] = "unknown"

news["relevance_score"] = pd.to_numeric(news["relevance_score"], errors="coerce").fillna(1)

print("News shape:", news.shape)
print(news["topic"].value_counts())

display(news[["published_utc", "source", "title_clean", "topic", "relevance_score"]].head())

News shape: (209923, 12)
topic
military_escalation    100012
hormuz_shipping         36761
other_oil_relevant      23241
macro_dollar_fed        19059
opec_gulf_supply        15310
inventory_eia            7490
sanctions_exports        3616
diplomacy_ceasefire      3261
demand_china_growth      1173
Name: count, dtype: int64


,published_utc,source,title_clean,topic,relevance_score
0,2026-02-15 00:00:00+00:00,masdark.com,2026 Winter Olympics: Mikhail Shaidorov wins m...,other_oil_relevant,1
1,2026-02-15 00:00:00+00:00,nwaonline.com,Olympic gold medalist Noah Lyles runs personal...,other_oil_relevant,1
2,2026-02-15 00:00:00+00:00,640wxsm.com,Charley Hull uses late charge to win PIF Saudi...,opec_gulf_supply,2
3,2026-02-15 00:00:00+00:00,valleynewslive.com,Jordan Stolz wins his 2nd Olympic speedskating...,other_oil_relevant,1
4,2026-02-15 00:00:00+00:00,fortworthbusiness.com,Fort Worth Chamber Summit to Address Workforce...,other_oil_relevant,2


In [11]:
def aggregate_text_features(df, freq="4h"):
    temp = df.copy()
    temp["bucket"] = temp["published_utc"].dt.floor(freq)

    all_agg = temp.groupby("bucket").agg(
        all_news_count=("title_clean", "size"),
        all_source_count=("source", "nunique"),
        all_relevance_sum=("relevance_score", "sum"),
        all_relevance_mean=("relevance_score", "mean")
    )

    topic_agg = temp.groupby(["bucket", "topic"]).agg(
        count=("title_clean", "size"),
        source_count=("source", "nunique"),
        relevance_sum=("relevance_score", "sum"),
        relevance_mean=("relevance_score", "mean")
    )

    topic_wide = topic_agg.unstack("topic")

    topic_wide.columns = [
        f"{topic}_{metric}"
        for metric, topic in topic_wide.columns
    ]

    features = all_agg.join(topic_wide, how="outer").fillna(0)

    # log1p features
    for c in list(features.columns):
        if c.endswith("_count") or c.endswith("_sum") or c in ["all_news_count", "all_relevance_sum"]:
            features[f"log1p_{c}"] = np.log1p(features[c])

    features.index.name = "timestamp_utc"
    features = features.reset_index()

    return features


text_4h = aggregate_text_features(news, freq="4h")

print("Text 4h shape:", text_4h.shape)
display(text_4h.head())

text_4h.to_csv(DATA_DIR / "text_features_4h_topic_count_only.csv", index=False)
print("Saved:", DATA_DIR / "text_features_4h_topic_count_only.csv")

Text 4h shape: (426, 71)


,timestamp_utc,all_news_count,all_source_count,all_relevance_sum,all_relevance_mean,demand_china_growth_count,diplomacy_ceasefire_count,hormuz_shipping_count,inventory_eia_count,macro_dollar_fed_count,...,log1p_sanctions_exports_source_count,log1p_demand_china_growth_relevance_sum,log1p_diplomacy_ceasefire_relevance_sum,log1p_hormuz_shipping_relevance_sum,log1p_inventory_eia_relevance_sum,log1p_macro_dollar_fed_relevance_sum,log1p_military_escalation_relevance_sum,log1p_opec_gulf_supply_relevance_sum,log1p_other_oil_relevant_relevance_sum,log1p_sanctions_exports_relevance_sum
0,2026-02-15 00:00:00+00:00,126,87,186,1.476190,5.0,1.0,2.0,1.0,13.0,...,2.079442,2.484907,1.386294,1.945910,1.098612,2.639057,3.806662,2.944439,4.262680,2.995732
1,2026-02-15 04:00:00+00:00,119,84,191,1.605042,2.0,0.0,3.0,5.0,27.0,...,1.791759,1.945910,0.000000,2.302585,2.397895,3.332205,3.871201,3.433987,3.806662,2.944439
2,2026-02-15 08:00:00+00:00,128,86,234,1.828125,5.0,2.0,3.0,3.0,14.0,...,2.302585,2.772589,1.609438,2.302585,1.945910,2.708050,4.007333,4.356709,3.401197,3.295837
3,2026-02-15 12:00:00+00:00,182,127,317,1.741758,2.0,4.0,16.0,5.0,25.0,...,2.079442,1.609438,2.079442,4.158883,2.397895,3.258097,3.637586,4.488636,4.158883,3.044522
4,2026-02-15 16:00:00+00:00,176,125,306,1.738636,0.0,2.0,22.0,2.0,22.0,...,1.098612,0.000000,1.609438,4.394449,1.609438,3.135494,4.060443,3.970292,4.406719,1.945910


Saved: /content/drive/MyDrive/bigquery/text_features_4h_topic_count_only.csv


In [12]:
tickers = {
    "WTI": "CL=F",
    "BRENT": "BZ=F",
    "GOLD": "GC=F",
    "NATGAS": "NG=F",
    "SPY": "SPY",
    "UUP": "UUP",
    "VIX": "^VIX"
}

raw = yf.download(
    list(tickers.values()),
    start=EVENT_START,
    end=EVENT_END,
    interval="1h",
    auto_adjust=False,
    progress=True,
    group_by="ticker",
    threads=True
)

print(raw.shape)
display(raw.head())

[*********************100%***********************]  7 of 7 completed

(1448, 42)


Ticker                    ^VIX                                  SPY           \
Price                     Open High Low Close Adj Close Volume Open High Low   
Datetime                                                                       
2026-02-15 23:00:00+00:00  NaN  NaN NaN   NaN       NaN    NaN  NaN  NaN NaN   
2026-02-16 00:00:00+00:00  NaN  NaN NaN   NaN       NaN    NaN  NaN  NaN NaN   
2026-02-16 01:00:00+00:00  NaN  NaN NaN   NaN       NaN    NaN  NaN  NaN NaN   
2026-02-16 02:00:00+00:00  NaN  NaN NaN   NaN       NaN    NaN  NaN  NaN NaN   
2026-02-16 03:00:00+00:00  NaN  NaN NaN   NaN       NaN    NaN  NaN  NaN NaN   

Ticker                           ...         GC=F                            \
Price                     Close  ...          Low        Close    Adj Close   
Datetime                         ...                                          
2026-02-15 23:00:00+00:00   NaN  ...  5031.600098  5038.500000  5038.500000   
2026-02-16 00:00:00+00:00   NaN  ...  5018.200195  5035.000000  5035.000000   
2026-02-16 01:00:00+00:00   NaN  ...  5026.399902  5047.600098  5047.600098   
2026-02-16 02:00:00+00:00   NaN  ...  5011.299805  5014.799805  5014.799805   
2026-02-16 03:00:00+00:00   NaN  ...  4986.000000  5007.200195  5007.200195   

Ticker                                  CL=F                                   \
Price                      Volume       Open       High        Low      Close   
Datetime                                                                        
2026-02-15 23:00:00+00:00   917.0  62.889999  62.900002  62.810001  62.900002   
2026-02-16 00:00:00+00:00  1713.0  62.900002  62.939999  62.830002  62.860001   
2026-02-16 01:00:00+00:00  1387.0  62.860001  62.930000  62.830002  62.869999   
2026-02-16 02:00:00+00:00  2108.0  62.860001  62.959999  62.759998  62.900002   
2026-02-16 03:00:00+00:00  2652.0  62.900002  62.970001  62.820000  62.910000   

Ticker                                        
Price                      Adj Close  Volume  
Datetime                                      
2026-02-15 23:00:00+00:00  62.900002  1005.0  
2026-02-16 00:00:00+00:00  62.860001   398.0  
2026-02-16 01:00:00+00:00  62.869999   522.0  
2026-02-16 02:00:00+00:00  62.900002   778.0  
2026-02-16 03:00:00+00:00  62.910000   976.0  

[5 rows x 42 columns]

In [13]:
def extract_close(data, ticker_map):
    close = pd.DataFrame()

    for name, ticker in ticker_map.items():
        try:
            if isinstance(data.columns, pd.MultiIndex):
                if (ticker, "Close") in data.columns:
                    close[name] = data[(ticker, "Close")]
                elif ("Close", ticker) in data.columns:
                    close[name] = data[("Close", ticker)]
            else:
                if "Close" in data.columns:
                    close[name] = data["Close"]
        except Exception as e:
            print("Cannot extract:", name, ticker, e)

    close.index = pd.to_datetime(close.index)

    if close.index.tz is None:
        close.index = close.index.tz_localize("UTC")
    else:
        close.index = close.index.tz_convert("UTC")

    return close.dropna(how="all")


px_1h = extract_close(raw, tickers)

print("Price 1h shape:", px_1h.shape)
display(px_1h.head())

px_1h.to_csv(DATA_DIR / "yahoo_market_prices_1h.csv")
print("Saved:", DATA_DIR / "yahoo_market_prices_1h.csv")

Price 1h shape: (1448, 7)


,WTI,BRENT,GOLD,NATGAS,SPY,UUP,VIX
Datetime,,,,,,,
2026-02-15 23:00:00+00:00,62.900002,67.739998,5038.500000,3.038,NaN,NaN,NaN
2026-02-16 00:00:00+00:00,62.860001,67.730003,5035.000000,3.016,NaN,NaN,NaN
2026-02-16 01:00:00+00:00,62.869999,67.690002,5047.600098,3.015,NaN,NaN,NaN
2026-02-16 02:00:00+00:00,62.900002,67.739998,5014.799805,3.011,NaN,NaN,NaN
2026-02-16 03:00:00+00:00,62.910000,67.820000,5007.200195,3.022,NaN,NaN,NaN


Saved: /content/drive/MyDrive/bigquery/yahoo_market_prices_1h.csv


In [14]:
def make_price_features(px, freq="4h"):
    p = px.copy()

    # 4h price: use last available price in each 4h window
    p = p.resample(freq, label="left", closed="left").last()
    p = p.dropna(subset=["WTI"]).ffill()

    feat = pd.DataFrame(index=p.index)
    feat["wti_close"] = p["WTI"]

    # WTI return
    feat["wti_ret_0"] = np.log(p["WTI"]).diff()

    # WTI lag returns
    for lag in [1, 2, 3, 6, 12]:
        feat[f"wti_ret_lag{lag}"] = feat["wti_ret_0"].shift(lag)

    # WTI rolling volatility
    for window in [3, 6, 12]:
        feat[f"wti_vol_{window}"] = feat["wti_ret_0"].rolling(window).std()

    # other market returns
    for col in p.columns:
        if col == "WTI":
            continue

        ret = np.log(p[col]).diff()
        name = col.lower()

        feat[f"{name}_ret_0"] = ret

        for lag in [1, 2, 3]:
            feat[f"{name}_ret_lag{lag}"] = ret.shift(lag)

    # calendar controls
    feat["hour"] = feat.index.hour
    feat["dayofweek"] = feat.index.dayofweek
    feat["hour_sin"] = np.sin(2 * np.pi * feat["hour"] / 24)
    feat["hour_cos"] = np.cos(2 * np.pi * feat["hour"] / 24)

    # target: next 4h return and direction
    feat["future_ret_next"] = np.log(p["WTI"].shift(-1)) - np.log(p["WTI"])
    feat["target_direction"] = (feat["future_ret_next"] > 0).astype(int)

    feat.index.name = "timestamp_utc"
    feat = feat.reset_index()

    return feat


price_4h = make_price_features(px_1h, freq="4h")

print("Price 4h shape:", price_4h.shape)
display(price_4h.head())

Price 4h shape: (298, 41)


,timestamp_utc,wti_close,wti_ret_0,wti_ret_lag1,wti_ret_lag2,wti_ret_lag3,wti_ret_lag6,wti_ret_lag12,wti_vol_3,wti_vol_6,...,vix_ret_0,vix_ret_lag1,vix_ret_lag2,vix_ret_lag3,hour,dayofweek,hour_sin,hour_cos,future_ret_next,target_direction
0,2026-02-15 20:00:00+00:00,62.900002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,20,6,-0.866025,0.5,0.000159,1
1,2026-02-16 00:00:00+00:00,62.910000,0.000159,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0,0,0.000000,1.0,0.000000,0
2,2026-02-16 04:00:00+00:00,62.910000,0.000000,0.000159,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,4,0,0.866025,0.5,0.004599,1
3,2026-02-17 04:00:00+00:00,63.200001,0.004599,0.000000,0.000159,NaN,NaN,NaN,0.002611,NaN,...,NaN,NaN,NaN,NaN,4,1,0.866025,0.5,0.007409,1
4,2026-02-17 08:00:00+00:00,63.669998,0.007409,0.004599,0.000000,0.000159,NaN,NaN,0.003740,NaN,...,NaN,NaN,NaN,NaN,8,1,0.866025,-0.5,-0.027870,0


In [15]:
text_4h["timestamp_utc"] = pd.to_datetime(text_4h["timestamp_utc"], utc=True)
price_4h["timestamp_utc"] = pd.to_datetime(price_4h["timestamp_utc"], utc=True)

df4 = price_4h.merge(text_4h, on="timestamp_utc", how="left")

text_cols_raw = [c for c in text_4h.columns if c != "timestamp_utc"]
df4[text_cols_raw] = df4[text_cols_raw].fillna(0)

df4 = df4.replace([np.inf, -np.inf], np.nan)

df4 = df4.dropna(subset=["future_ret_next", "target_direction"]).copy()

print("Final 4h dataset shape:", df4.shape)
print("Target distribution:")
print(df4["target_direction"].value_counts(normalize=True))

display(df4.head())

df4.to_csv(DATA_DIR / "direction_dataset_4h_topic_count_only.csv", index=False)
print("Saved:", DATA_DIR / "direction_dataset_4h_topic_count_only.csv")

Final 4h dataset shape: (297, 111)
Target distribution:
target_direction
1    0.575758
0    0.424242
Name: proportion, dtype: float64


,timestamp_utc,wti_close,wti_ret_0,wti_ret_lag1,wti_ret_lag2,wti_ret_lag3,wti_ret_lag6,wti_ret_lag12,wti_vol_3,wti_vol_6,...,log1p_sanctions_exports_source_count,log1p_demand_china_growth_relevance_sum,log1p_diplomacy_ceasefire_relevance_sum,log1p_hormuz_shipping_relevance_sum,log1p_inventory_eia_relevance_sum,log1p_macro_dollar_fed_relevance_sum,log1p_military_escalation_relevance_sum,log1p_opec_gulf_supply_relevance_sum,log1p_other_oil_relevant_relevance_sum,log1p_sanctions_exports_relevance_sum
0,2026-02-15 20:00:00+00:00,62.900002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000000,1.609438,1.386294,4.770685,2.708050,3.218876,3.713572,3.433987,4.248495,0.000000
1,2026-02-16 00:00:00+00:00,62.910000,0.000159,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.098612,1.386294,2.197225,4.828314,2.197225,3.806662,3.688879,3.871201,4.007333,1.609438
2,2026-02-16 04:00:00+00:00,62.910000,0.000000,0.000159,NaN,NaN,NaN,NaN,NaN,NaN,...,1.791759,1.098612,2.708050,3.583519,4.574711,3.970292,3.218876,4.204693,4.369448,2.639057
3,2026-02-17 04:00:00+00:00,63.200001,0.004599,0.000000,0.000159,NaN,NaN,NaN,0.002611,NaN,...,1.945910,1.945910,2.944439,3.688879,3.091042,4.025352,3.218876,4.007333,4.465908,2.708050
4,2026-02-17 08:00:00+00:00,63.669998,0.007409,0.004599,0.000000,0.000159,NaN,NaN,0.003740,NaN,...,2.197225,1.609438,3.258097,4.927254,2.833213,4.094345,4.499810,4.663439,4.304065,2.890372


Saved: /content/drive/MyDrive/bigquery/direction_dataset_4h_topic_count_only.csv


In [16]:
core_topics = [
    "inventory_eia",
    "diplomacy_ceasefire",
    "military_escalation",
    "hormuz_shipping",
    "sanctions_exports"
]

exclude_cols = [
    "timestamp_utc",
    "wti_close",
    "future_ret_next",
    "target_direction"
]

numeric_cols = [
    c for c in df4.columns
    if c not in exclude_cols
    and pd.api.types.is_numeric_dtype(df4[c])
]

def is_core_topic_feature(col):
    for topic in core_topics:
        if col.startswith(topic + "_") or col.startswith("log1p_" + topic + "_"):
            return True
    return False

core_text_cols = [
    c for c in numeric_cols
    if is_core_topic_feature(c)
]

# price / financial features
# 这里排除所有非核心 topic，只保留价格和金融变量
non_core_topic_prefixes = [
    "all_",
    "log1p_all_",
    "other_oil_relevant_",
    "log1p_other_oil_relevant_",
    "macro_dollar_fed_",
    "log1p_macro_dollar_fed_",
    "demand_china_growth_",
    "log1p_demand_china_growth_",
    "opec_gulf_supply_",
    "log1p_opec_gulf_supply_",
]

financial_cols = [
    c for c in numeric_cols
    if c not in core_text_cols
    and not any(c.startswith(prefix) for prefix in non_core_topic_prefixes)
]

print("Core text features:", len(core_text_cols))
print(core_text_cols)

print("\nFinancial features:", len(financial_cols))
print(financial_cols[:30])

Core text features: 35
['diplomacy_ceasefire_count', 'hormuz_shipping_count', 'inventory_eia_count', 'military_escalation_count', 'sanctions_exports_count', 'diplomacy_ceasefire_source_count', 'hormuz_shipping_source_count', 'inventory_eia_source_count', 'military_escalation_source_count', 'sanctions_exports_source_count', 'diplomacy_ceasefire_relevance_sum', 'hormuz_shipping_relevance_sum', 'inventory_eia_relevance_sum', 'military_escalation_relevance_sum', 'sanctions_exports_relevance_sum', 'diplomacy_ceasefire_relevance_mean', 'hormuz_shipping_relevance_mean', 'inventory_eia_relevance_mean', 'military_escalation_relevance_mean', 'sanctions_exports_relevance_mean', 'log1p_diplomacy_ceasefire_count', 'log1p_hormuz_shipping_count', 'log1p_inventory_eia_count', 'log1p_military_escalation_count', 'log1p_sanctions_exports_count', 'log1p_diplomacy_ceasefire_source_count', 'log1p_hormuz_shipping_source_count', 'log1p_inventory_eia_source_count', 'log1p_military_escalation_source_count', 'lo

In [17]:
df4_core = df4.sort_values("timestamp_utc").reset_index(drop=True).copy()

lag_base_cols = [
    c for c in core_text_cols
    if (
        c.endswith("_count")
        or c.endswith("_source_count")
        or c.endswith("_relevance_sum")
        or c.startswith("log1p_")
    )
]

for col in lag_base_cols:
    for lag in [1, 2, 3]:
        df4_core[f"{col}_lag{lag}"] = df4_core[col].shift(lag)

lag_cols = [
    c for c in df4_core.columns
    if c.endswith("_lag1") or c.endswith("_lag2") or c.endswith("_lag3")
]

df4_core[lag_cols] = df4_core[lag_cols].fillna(0)

print("Current core features:", len(core_text_cols))
print("Lag features:", len(lag_cols))
print("Total core + lag:", len(core_text_cols + lag_cols))

Current core features: 35
Lag features: 111
Total core + lag: 146


In [18]:
feature_sets = {
    "Baseline": [],
    "Core Text current": core_text_cols,
    "Core Text current+lag": core_text_cols + lag_cols,
    "Financial-only": financial_cols,
    "Financial + Core Text current": financial_cols + core_text_cols,
    "Financial + Core Text current+lag": financial_cols + core_text_cols + lag_cols,
}

for name, cols in feature_sets.items():
    print(name, len(cols))

Baseline 0
Core Text current 35
Core Text current+lag 146
Financial-only 37
Financial + Core Text current 72
Financial + Core Text current+lag 183


In [19]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

from xgboost import XGBClassifier


def find_best_threshold(y_true, prob, metric="balanced_accuracy"):
    thresholds = np.linspace(0.30, 0.70, 41)
    best_t = 0.50
    best_score = -999

    for t in thresholds:
        pred = (prob >= t).astype(int)

        if metric == "accuracy":
            score = accuracy_score(y_true, pred)
        else:
            score = balanced_accuracy_score(y_true, pred)

        if score > best_score:
            best_score = score
            best_t = t

    return best_t


def make_model(model_name):
    if model_name == "Baseline":
        return DummyClassifier(strategy="most_frequent")

    if model_name.startswith("Logit_L1"):
        C = float(model_name.split("_C")[-1])
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                penalty="l1",
                C=C,
                solver="liblinear",
                max_iter=5000,
                class_weight="balanced"
            ))
        ])

    if model_name.startswith("Logit_L2"):
        C = float(model_name.split("_C")[-1])
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                penalty="l2",
                C=C,
                solver="lbfgs",
                max_iter=5000,
                class_weight="balanced"
            ))
        ])

    if model_name.startswith("XGB"):
        parts = model_name.split("_")
        depth = int(parts[1].replace("d", ""))
        lr = float(parts[2].replace("lr", ""))
        n_est = int(parts[3].replace("n", ""))

        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", XGBClassifier(
                n_estimators=n_est,
                max_depth=depth,
                learning_rate=lr,
                subsample=0.8,
                colsample_bytree=0.8,
                eval_metric="logloss",
                random_state=42,
                n_jobs=-1
            ))
        ])

    raise ValueError(model_name)


def evaluate_walkforward_direction(
    df,
    feature_cols,
    model_name,
    threshold_metric="balanced_accuracy",
    n_splits=5
):
    data = df.sort_values("timestamp_utc").reset_index(drop=True).copy()

    y = data["target_direction"].astype(int).values

    if len(feature_cols) == 0:
        X = pd.DataFrame({"constant": np.zeros(len(data))})
    else:
        X = data[feature_cols].replace([np.inf, -np.inf], np.nan)

    tscv = TimeSeriesSplit(n_splits=n_splits)

    y_true_all = []
    y_pred_all = []
    y_prob_all = []
    thresholds = []

    for train_idx, test_idx in tscv.split(X):
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]
        y_train = y[train_idx]
        y_test = y[test_idx]

        if len(np.unique(y_train)) < 2:
            continue

        model = make_model(model_name)
        model.fit(X_train, y_train)

        if model_name == "Baseline":
            test_pred = model.predict(X_test)
            test_prob = test_pred
            t = np.nan
        else:
            train_prob = model.predict_proba(X_train)[:, 1]
            test_prob = model.predict_proba(X_test)[:, 1]

            t = find_best_threshold(y_train, train_prob, metric=threshold_metric)
            test_pred = (test_prob >= t).astype(int)

        y_true_all.extend(y_test)
        y_pred_all.extend(test_pred)
        y_prob_all.extend(test_prob)
        thresholds.append(t)

    y_true_all = np.array(y_true_all)
    y_pred_all = np.array(y_pred_all)
    y_prob_all = np.array(y_prob_all)

    try:
        auc = roc_auc_score(y_true_all, y_prob_all)
    except:
        auc = np.nan

    return {
        "model": model_name,
        "n_features": len(feature_cols),
        "accuracy": accuracy_score(y_true_all, y_pred_all),
        "balanced_accuracy": balanced_accuracy_score(y_true_all, y_pred_all),
        "f1": f1_score(y_true_all, y_pred_all, zero_division=0),
        "auc": auc,
        "up_rate_true": y_true_all.mean(),
        "up_rate_pred": y_pred_all.mean(),
        "avg_threshold": np.nanmean(thresholds),
        "n_eval": len(y_true_all),
        "confusion_matrix": confusion_matrix(y_true_all, y_pred_all).tolist()
    }

In [20]:
models = []

for C in [0.01, 0.05, 0.1, 0.5, 1.0, 2.0]:
    models.append(f"Logit_L1_C{C}")
    models.append(f"Logit_L2_C{C}")

for d in [1, 2]:
    for lr in [0.01, 0.03, 0.05]:
        for n in [100, 200]:
            models.append(f"XGB_d{d}_lr{lr}_n{n}")

rows = []

for feature_set_name, cols in feature_sets.items():
    if feature_set_name == "Baseline":
        run_models = ["Baseline"]
    else:
        run_models = models

    for model_name in run_models:
        print(f"Running | {feature_set_name} | {model_name} | n_features={len(cols)}")

        try:
            res = evaluate_walkforward_direction(
                df4_core,
                feature_cols=cols,
                model_name=model_name,
                threshold_metric="balanced_accuracy",
                n_splits=5
            )

            res["feature_set"] = feature_set_name
            rows.append(res)

        except Exception as e:
            print("Failed:", feature_set_name, model_name, e)

core_results = pd.DataFrame(rows)

core_results = core_results.sort_values(
    ["balanced_accuracy", "auc", "accuracy"],
    ascending=False
)

display(
    core_results[
        [
            "feature_set",
            "model",
            "n_features",
            "accuracy",
            "balanced_accuracy",
            "f1",
            "auc",
            "up_rate_true",
            "up_rate_pred",
            "avg_threshold"
        ]
    ].head(30)
)

core_results.to_csv(DATA_DIR / "core_topic_only_4h_results.csv", index=False)
print("Saved:", DATA_DIR / "core_topic_only_4h_results.csv")

Running | Baseline | Baseline | n_features=0
Running | Core Text current | Logit_L1_C0.01 | n_features=35
Running | Core Text current | Logit_L2_C0.01 | n_features=35
Running | Core Text current | Logit_L1_C0.05 | n_features=35
Running | Core Text current | Logit_L2_C0.05 | n_features=35
Running | Core Text current | Logit_L1_C0.1 | n_features=35
Running | Core Text current | Logit_L2_C0.1 | n_features=35
Running | Core Text current | Logit_L1_C0.5 | n_features=35
Running | Core Text current | Logit_L2_C0.5 | n_features=35
Running | Core Text current | Logit_L1_C1.0 | n_features=35
Running | Core Text current | Logit_L2_C1.0 | n_features=35
Running | Core Text current | Logit_L1_C2.0 | n_features=35
Running | Core Text current | Logit_L2_C2.0 | n_features=35
Running | Core Text current | XGB_d1_lr0.01_n100 | n_features=35
Running | Core Text current | XGB_d1_lr0.01_n200 | n_features=35
Running | Core Text current | XGB_d1_lr0.03_n100 | n_features=35
Running | Core Text current | XGB_d1

,feature_set,model,n_features,accuracy,balanced_accuracy,f1,auc,up_rate_true,up_rate_pred,avg_threshold
7,Core Text current,Logit_L1_C0.5,35,0.575510,0.535862,0.677019,0.560828,0.591837,0.722449,0.486
4,Core Text current,Logit_L2_C0.05,35,0.591837,0.534138,0.710983,0.531172,0.591837,0.820408,0.488
6,Core Text current,Logit_L2_C0.1,35,0.583673,0.525690,0.705202,0.525103,0.591837,0.820408,0.484
2,Core Text current,Logit_L2_C0.01,35,0.579592,0.525345,0.697947,0.547586,0.591837,0.800000,0.492
3,Core Text current,Logit_L1_C0.05,35,0.600000,0.517759,0.740741,0.505448,0.591837,0.951020,0.340
27,Core Text current+lag,Logit_L1_C0.05,146,0.600000,0.517759,0.740741,0.505448,0.591837,0.951020,0.340
5,Core Text current,Logit_L1_C0.1,35,0.583673,0.513276,0.718232,0.528241,0.591837,0.885714,0.436
29,Core Text current+lag,Logit_L1_C0.1,146,0.579592,0.512931,0.711485,0.521000,0.591837,0.865306,0.442
75,Financial + Core Text current,Logit_L1_C0.05,72,0.595918,0.512759,0.738786,0.496862,0.591837,0.955102,0.340
99,Financial + Core Text current+lag,Logit_L1_C0.05,183,0.595918,0.512759,0.738786,0.496862,0.591837,0.955102,0.340


Saved: /content/drive/MyDrive/bigquery/core_topic_only_4h_results.csv


In [21]:
display(
    core_results[
        [
            "feature_set",
            "model",
            "n_features",
            "accuracy",
            "balanced_accuracy",
            "f1",
            "auc",
            "up_rate_true",
            "up_rate_pred",
            "avg_threshold"
        ]
    ].sort_values(["accuracy", "balanced_accuracy", "auc"], ascending=False).head(30)
)

,feature_set,model,n_features,accuracy,balanced_accuracy,f1,auc,up_rate_true,up_rate_pred,avg_threshold
3,Core Text current,Logit_L1_C0.05,35,0.600000,0.517759,0.740741,0.505448,0.591837,0.951020,0.340
27,Core Text current+lag,Logit_L1_C0.05,146,0.600000,0.517759,0.740741,0.505448,0.591837,0.951020,0.340
75,Financial + Core Text current,Logit_L1_C0.05,72,0.595918,0.512759,0.738786,0.496862,0.591837,0.955102,0.340
99,Financial + Core Text current+lag,Logit_L1_C0.05,183,0.595918,0.512759,0.738786,0.496862,0.591837,0.955102,0.340
4,Core Text current,Logit_L2_C0.05,35,0.591837,0.534138,0.710983,0.531172,0.591837,0.820408,0.488
1,Core Text current,Logit_L1_C0.01,35,0.591837,0.500000,0.743590,0.500000,0.591837,1.000000,0.300
25,Core Text current+lag,Logit_L1_C0.01,146,0.591837,0.500000,0.743590,0.500000,0.591837,1.000000,0.300
49,Financial-only,Logit_L1_C0.01,37,0.591837,0.500000,0.743590,0.500000,0.591837,1.000000,0.300
73,Financial + Core Text current,Logit_L1_C0.01,72,0.591837,0.500000,0.743590,0.500000,0.591837,1.000000,0.300
97,Financial + Core Text current+lag,Logit_L1_C0.01,183,0.591837,0.500000,0.743590,0.500000,0.591837,1.000000,0.300


In [23]:
!pip install -q xgboost lightgbm

In [24]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import TimeSeriesSplit
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [25]:
feature_sets_tree = {
    "Core Text current": core_text_cols,
    "Core Text current+lag": core_text_cols + lag_cols,
}

# 如果你还想对比 financial，就保留下面这几组
if "financial_cols" in globals():
    feature_sets_tree.update({
        "Financial-only": financial_cols,
        "Financial + Core Text current": financial_cols + core_text_cols,
        "Financial + Core Text current+lag": financial_cols + core_text_cols + lag_cols,
    })

for name, cols in feature_sets_tree.items():
    print(name, len(cols))

Core Text current 35
Core Text current+lag 146
Financial-only 37
Financial + Core Text current 72
Financial + Core Text current+lag 183


In [26]:
def make_tree_model(model_name):
    if model_name.startswith("XGB"):
        parts = model_name.split("_")
        depth = int(parts[1].replace("d", ""))
        lr = float(parts[2].replace("lr", ""))
        n_est = int(parts[3].replace("n", ""))

        return XGBClassifier(
            n_estimators=n_est,
            max_depth=depth,
            learning_rate=lr,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_weight=3,
            reg_alpha=0.1,
            reg_lambda=1.0,
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        )

    if model_name.startswith("RF"):
        return RandomForestClassifier(
            n_estimators=300,
            max_depth=3,
            min_samples_leaf=5,
            max_features="sqrt",
            random_state=42,
            class_weight=None,
            n_jobs=-1
        )

    if model_name.startswith("GB"):
        return GradientBoostingClassifier(
            n_estimators=100,
            max_depth=2,
            learning_rate=0.03,
            random_state=42
        )

    if model_name.startswith("LGBM"):
        return LGBMClassifier(
            n_estimators=150,
            max_depth=2,
            learning_rate=0.03,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=1.0,
            random_state=42,
            verbose=-1
        )

    raise ValueError(model_name)

In [27]:
def find_best_threshold(y_true, prob, metric="accuracy"):
    thresholds = np.linspace(0.20, 0.80, 61)
    best_t = 0.50
    best_score = -999

    for t in thresholds:
        pred = (prob >= t).astype(int)

        if metric == "accuracy":
            score = accuracy_score(y_true, pred)
        elif metric == "balanced_accuracy":
            score = balanced_accuracy_score(y_true, pred)
        else:
            raise ValueError("metric must be accuracy or balanced_accuracy")

        if score > best_score:
            best_score = score
            best_t = t

    return best_t

In [28]:
def evaluate_tree_walkforward(
    df,
    feature_cols,
    model_name,
    threshold_metric="accuracy",
    n_splits=5
):
    data = df.sort_values("timestamp_utc").reset_index(drop=True).copy()

    X_all = data[feature_cols].replace([np.inf, -np.inf], np.nan)
    y_all = data["target_direction"].astype(int).values

    tscv = TimeSeriesSplit(n_splits=n_splits)

    y_true_all = []
    y_pred_all = []
    y_prob_all = []
    thresholds = []

    for train_idx, test_idx in tscv.split(X_all):
        X_train = X_all.iloc[train_idx]
        X_test = X_all.iloc[test_idx]

        y_train = y_all[train_idx]
        y_test = y_all[test_idx]

        if len(np.unique(y_train)) < 2:
            continue

        imputer = SimpleImputer(strategy="median")
        X_train_imp = imputer.fit_transform(X_train)
        X_test_imp = imputer.transform(X_test)

        model = make_tree_model(model_name)
        model.fit(X_train_imp, y_train)

        train_prob = model.predict_proba(X_train_imp)[:, 1]
        test_prob = model.predict_proba(X_test_imp)[:, 1]

        threshold = find_best_threshold(
            y_train,
            train_prob,
            metric=threshold_metric
        )

        test_pred = (test_prob >= threshold).astype(int)

        y_true_all.extend(y_test)
        y_pred_all.extend(test_pred)
        y_prob_all.extend(test_prob)
        thresholds.append(threshold)

    y_true_all = np.array(y_true_all)
    y_pred_all = np.array(y_pred_all)
    y_prob_all = np.array(y_prob_all)

    try:
        auc = roc_auc_score(y_true_all, y_prob_all)
    except:
        auc = np.nan

    return {
        "model": model_name,
        "n_features": len(feature_cols),
        "accuracy": accuracy_score(y_true_all, y_pred_all),
        "balanced_accuracy": balanced_accuracy_score(y_true_all, y_pred_all),
        "f1": f1_score(y_true_all, y_pred_all, zero_division=0),
        "auc": auc,
        "up_rate_true": y_true_all.mean(),
        "up_rate_pred": y_pred_all.mean(),
        "avg_threshold": np.mean(thresholds),
        "n_eval": len(y_true_all),
        "confusion_matrix": confusion_matrix(y_true_all, y_pred_all).tolist()
    }

In [29]:
tree_models = []

# XGBoost: shallow trees
for d in [1, 2, 3]:
    for lr in [0.01, 0.03, 0.05, 0.08]:
        for n in [50, 100, 200]:
            tree_models.append(f"XGB_d{d}_lr{lr}_n{n}")

# Random Forest
tree_models.append("RF")

# Gradient Boosting
tree_models.append("GB")

# LightGBM
tree_models.append("LGBM")

print("Number of tree models:", len(tree_models))

Number of tree models: 39


In [30]:
tree_rows = []

for feature_set_name, cols in feature_sets_tree.items():
    for model_name in tree_models:
        print(f"Running | {feature_set_name} | {model_name} | n_features={len(cols)}")

        try:
            res = evaluate_tree_walkforward(
                df4_core,
                feature_cols=cols,
                model_name=model_name,
                threshold_metric="accuracy",   # 如果想看 balanced accuracy，改成 balanced_accuracy
                n_splits=5
            )

            res["feature_set"] = feature_set_name
            tree_rows.append(res)

        except Exception as e:
            print("Failed:", feature_set_name, model_name, e)

tree_results = pd.DataFrame(tree_rows)

tree_results = tree_results.sort_values(
    ["accuracy", "balanced_accuracy", "auc"],
    ascending=False
)

display(
    tree_results[
        [
            "feature_set",
            "model",
            "n_features",
            "accuracy",
            "balanced_accuracy",
            "f1",
            "auc",
            "up_rate_true",
            "up_rate_pred",
            "avg_threshold"
        ]
    ].head(30)
)

tree_results.to_csv(
    DATA_DIR / "tree_models_core_topic_4h_results.csv",
    index=False
)

print("Saved:", DATA_DIR / "tree_models_core_topic_4h_results.csv")

Running | Core Text current | XGB_d1_lr0.01_n50 | n_features=35
Running | Core Text current | XGB_d1_lr0.01_n100 | n_features=35
Running | Core Text current | XGB_d1_lr0.01_n200 | n_features=35
Running | Core Text current | XGB_d1_lr0.03_n50 | n_features=35
Running | Core Text current | XGB_d1_lr0.03_n100 | n_features=35
Running | Core Text current | XGB_d1_lr0.03_n200 | n_features=35
Running | Core Text current | XGB_d1_lr0.05_n50 | n_features=35
Running | Core Text current | XGB_d1_lr0.05_n100 | n_features=35
Running | Core Text current | XGB_d1_lr0.05_n200 | n_features=35
Running | Core Text current | XGB_d1_lr0.08_n50 | n_features=35
Running | Core Text current | XGB_d1_lr0.08_n100 | n_features=35
Running | Core Text current | XGB_d1_lr0.08_n200 | n_features=35
Running | Core Text current | XGB_d2_lr0.01_n50 | n_features=35
Running | Core Text current | XGB_d2_lr0.01_n100 | n_features=35
Running | Core Text current | XGB_d2_lr0.01_n200 | n_features=35
Running | Core Text current | 

,feature_set,model,n_features,accuracy,balanced_accuracy,f1,auc,up_rate_true,up_rate_pred,avg_threshold
194,Financial + Core Text current+lag,LGBM,183,0.563265,0.499138,0.696884,0.400069,0.591837,0.848980,0.508
188,Financial + Core Text current+lag,XGB_d3_lr0.05_n200,183,0.563265,0.486724,0.710027,0.397103,0.591837,0.914286,0.394
162,Financial + Core Text current+lag,XGB_d1_lr0.05_n50,183,0.559184,0.487931,0.701657,0.383931,0.591837,0.885714,0.526
191,Financial + Core Text current+lag,XGB_d3_lr0.08_n200,183,0.559184,0.481724,0.708108,0.397034,0.591837,0.918367,0.340
189,Financial + Core Text current+lag,XGB_d3_lr0.08_n50,183,0.551020,0.482586,0.692737,0.394621,0.591837,0.869388,0.496
85,Financial-only,XGB_d1_lr0.05_n100,37,0.546939,0.486897,0.680115,0.408552,0.591837,0.824490,0.496
117,Financial + Core Text current,XGB_d1_lr0.01_n50,72,0.546939,0.482241,0.685552,0.446552,0.591837,0.848980,0.548
24,Core Text current,XGB_d3_lr0.01_n50,35,0.542857,0.498966,0.656442,0.478655,0.591837,0.738776,0.550
84,Financial-only,XGB_d1_lr0.05_n50,37,0.542857,0.492759,0.664671,0.418931,0.591837,0.771429,0.538
10,Core Text current,XGB_d1_lr0.08_n100,35,0.542857,0.486552,0.672515,0.470897,0.591837,0.804082,0.498


Saved: /content/drive/MyDrive/bigquery/tree_models_core_topic_4h_results.csv


In [31]:
display(
    tree_results[
        [
            "feature_set",
            "model",
            "n_features",
            "accuracy",
            "balanced_accuracy",
            "f1",
            "auc",
            "up_rate_true",
            "up_rate_pred",
            "avg_threshold"
        ]
    ]
    .sort_values(["balanced_accuracy", "auc", "accuracy"], ascending=False)
    .head(30)
)

,feature_set,model,n_features,accuracy,balanced_accuracy,f1,auc,up_rate_true,up_rate_pred,avg_threshold
38,Core Text current,LGBM,35,0.538776,0.509483,0.631922,0.508310,0.591837,0.661224,0.530
194,Financial + Core Text current+lag,LGBM,183,0.563265,0.499138,0.696884,0.400069,0.591837,0.848980,0.508
24,Core Text current,XGB_d3_lr0.01_n50,35,0.542857,0.498966,0.656442,0.478655,0.591837,0.738776,0.550
84,Financial-only,XGB_d1_lr0.05_n50,37,0.542857,0.492759,0.664671,0.418931,0.591837,0.771429,0.538
87,Financial-only,XGB_d1_lr0.08_n50,37,0.538776,0.492414,0.656535,0.414517,0.591837,0.751020,0.520
96,Financial-only,XGB_d2_lr0.05_n50,37,0.538776,0.492414,0.656535,0.412414,0.591837,0.751020,0.546
162,Financial + Core Text current+lag,XGB_d1_lr0.05_n50,183,0.559184,0.487931,0.701657,0.383931,0.591837,0.885714,0.526
78,Financial-only,XGB_d1_lr0.01_n50,37,0.518367,0.487586,0.616883,0.451138,0.591837,0.665306,0.558
62,Core Text current+lag,XGB_d2_lr0.08_n200,146,0.534694,0.487414,0.654545,0.460276,0.591837,0.755102,0.450
85,Financial-only,XGB_d1_lr0.05_n100,37,0.546939,0.486897,0.680115,0.408552,0.591837,0.824490,0.496


In [32]:
xgb_results = tree_results[tree_results["model"].str.startswith("XGB")].copy()

display(
    xgb_results[
        [
            "feature_set",
            "model",
            "n_features",
            "accuracy",
            "balanced_accuracy",
            "f1",
            "auc",
            "up_rate_true",
            "up_rate_pred",
            "avg_threshold"
        ]
    ]
    .sort_values(["accuracy", "balanced_accuracy", "auc"], ascending=False)
    .head(20)
)

,feature_set,model,n_features,accuracy,balanced_accuracy,f1,auc,up_rate_true,up_rate_pred,avg_threshold
188,Financial + Core Text current+lag,XGB_d3_lr0.05_n200,183,0.563265,0.486724,0.710027,0.397103,0.591837,0.914286,0.394
162,Financial + Core Text current+lag,XGB_d1_lr0.05_n50,183,0.559184,0.487931,0.701657,0.383931,0.591837,0.885714,0.526
191,Financial + Core Text current+lag,XGB_d3_lr0.08_n200,183,0.559184,0.481724,0.708108,0.397034,0.591837,0.918367,0.340
189,Financial + Core Text current+lag,XGB_d3_lr0.08_n50,183,0.551020,0.482586,0.692737,0.394621,0.591837,0.869388,0.496
85,Financial-only,XGB_d1_lr0.05_n100,37,0.546939,0.486897,0.680115,0.408552,0.591837,0.824490,0.496
117,Financial + Core Text current,XGB_d1_lr0.01_n50,72,0.546939,0.482241,0.685552,0.446552,0.591837,0.848980,0.548
24,Core Text current,XGB_d3_lr0.01_n50,35,0.542857,0.498966,0.656442,0.478655,0.591837,0.738776,0.550
84,Financial-only,XGB_d1_lr0.05_n50,37,0.542857,0.492759,0.664671,0.418931,0.591837,0.771429,0.538
10,Core Text current,XGB_d1_lr0.08_n100,35,0.542857,0.486552,0.672515,0.470897,0.591837,0.804082,0.498
177,Financial + Core Text current+lag,XGB_d2_lr0.08_n50,183,0.542857,0.477241,0.683616,0.390759,0.591837,0.853061,0.510


In [33]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)
from xgboost import XGBClassifier
import pandas as pd
import numpy as np


def find_best_threshold_acc(y_true, prob):
    thresholds = np.linspace(0.10, 0.90, 81)
    best_t = 0.50
    best_acc = -1

    for t in thresholds:
        pred = (prob >= t).astype(int)
        acc = accuracy_score(y_true, pred)

        if acc > best_acc:
            best_acc = acc
            best_t = t

    return best_t


def make_accuracy_model(model_name):
    if model_name.startswith("Logit_L1"):
        C = float(model_name.split("_C")[-1])
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                penalty="l1",
                C=C,
                solver="liblinear",
                max_iter=5000,
                class_weight=None
            ))
        ])

    if model_name.startswith("Logit_L2"):
        C = float(model_name.split("_C")[-1])
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                penalty="l2",
                C=C,
                solver="lbfgs",
                max_iter=5000,
                class_weight=None
            ))
        ])

    if model_name.startswith("XGB"):
        parts = model_name.split("_")
        depth = int(parts[1].replace("d", ""))
        lr = float(parts[2].replace("lr", ""))
        n_est = int(parts[3].replace("n", ""))

        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", XGBClassifier(
                n_estimators=n_est,
                max_depth=depth,
                learning_rate=lr,
                subsample=0.8,
                colsample_bytree=0.8,
                eval_metric="logloss",
                random_state=42,
                n_jobs=-1
            ))
        ])

    raise ValueError(model_name)

In [34]:
# 只用核心 topic current features
# 如果你也想测试 lag，可以一起放进去

feature_sets_acc = {
    "Core Text current": core_text_cols,
    "Core Text current+lag": core_text_cols + lag_cols,
}

# 如果你还想对比金融变量，取消下面注释
feature_sets_acc.update({
    "Financial-only": financial_cols,
    "Financial + Core Text current": financial_cols + core_text_cols,
    "Financial + Core Text current+lag": financial_cols + core_text_cols + lag_cols,
})

for name, cols in feature_sets_acc.items():
    print(name, len(cols))

Core Text current 35
Core Text current+lag 146
Financial-only 37
Financial + Core Text current 72
Financial + Core Text current+lag 183


In [35]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

def evaluate_accuracy_logistic_walkforward(
    df,
    feature_cols,
    model_name,
    n_splits=5
):
    data = df.sort_values("timestamp_utc").reset_index(drop=True).copy()

    X = data[feature_cols].replace([np.inf, -np.inf], np.nan)
    y = data["target_direction"].astype(int).values

    tscv = TimeSeriesSplit(n_splits=n_splits)

    y_true_all = []
    y_pred_all = []
    y_prob_all = []
    threshold_list = []

    for train_idx, test_idx in tscv.split(X):
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]

        y_train = y[train_idx]
        y_test = y[test_idx]

        if len(np.unique(y_train)) < 2:
            continue

        model = make_accuracy_model(model_name)
        model.fit(X_train, y_train)

        train_prob = model.predict_proba(X_train)[:, 1]
        test_prob = model.predict_proba(X_test)[:, 1]

        # 在训练集上选择最大化 accuracy 的 threshold
        best_threshold = find_best_threshold_acc(y_train, train_prob)

        test_pred = (test_prob >= best_threshold).astype(int)

        y_true_all.extend(y_test)
        y_pred_all.extend(test_pred)
        y_prob_all.extend(test_prob)
        threshold_list.append(best_threshold)

    y_true_all = np.array(y_true_all)
    y_pred_all = np.array(y_pred_all)
    y_prob_all = np.array(y_prob_all)

    try:
        auc = roc_auc_score(y_true_all, y_prob_all)
    except:
        auc = np.nan

    return {
        "model": model_name,
        "n_features": len(feature_cols),
        "accuracy": accuracy_score(y_true_all, y_pred_all),
        "balanced_accuracy": balanced_accuracy_score(y_true_all, y_pred_all),
        "f1": f1_score(y_true_all, y_pred_all, zero_division=0),
        "auc": auc,
        "up_rate_true": y_true_all.mean(),
        "up_rate_pred": y_pred_all.mean(),
        "avg_threshold": np.mean(threshold_list),
        "n_eval": len(y_true_all),
        "confusion_matrix": confusion_matrix(y_true_all, y_pred_all).tolist()
    }

In [36]:
def evaluate_majority_baseline(df, n_splits=5):
    data = df.sort_values("timestamp_utc").reset_index(drop=True).copy()

    y = data["target_direction"].astype(int).values
    X_dummy = np.zeros((len(data), 1))

    tscv = TimeSeriesSplit(n_splits=n_splits)

    y_true_all = []
    y_pred_all = []

    for train_idx, test_idx in tscv.split(X_dummy):
        y_train = y[train_idx]
        y_test = y[test_idx]

        majority_class = int(pd.Series(y_train).mode().iloc[0])
        pred = np.repeat(majority_class, len(y_test))

        y_true_all.extend(y_test)
        y_pred_all.extend(pred)

    y_true_all = np.array(y_true_all)
    y_pred_all = np.array(y_pred_all)

    return {
        "feature_set": "Baseline",
        "model": "Majority",
        "n_features": 0,
        "accuracy": accuracy_score(y_true_all, y_pred_all),
        "balanced_accuracy": balanced_accuracy_score(y_true_all, y_pred_all),
        "f1": f1_score(y_true_all, y_pred_all, zero_division=0),
        "auc": 0.5,
        "up_rate_true": y_true_all.mean(),
        "up_rate_pred": y_pred_all.mean(),
        "avg_threshold": np.nan,
        "n_eval": len(y_true_all),
        "confusion_matrix": confusion_matrix(y_true_all, y_pred_all).tolist()
    }

In [37]:
logistic_models_acc = []

for C in [0.001, 0.005, 0.01, 0.03, 0.05, 0.1, 0.3, 0.5, 1.0, 2.0, 5.0]:
    logistic_models_acc.append(f"Logit_L1_C{C}")
    logistic_models_acc.append(f"Logit_L2_C{C}")

rows_acc_logit = []

# baseline
rows_acc_logit.append(evaluate_majority_baseline(df4_core, n_splits=5))

for feature_set_name, cols in feature_sets_acc.items():
    for model_name in logistic_models_acc:
        print(f"Running | {feature_set_name} | {model_name} | n_features={len(cols)}")

        try:
            res = evaluate_accuracy_logistic_walkforward(
                df4_core,
                feature_cols=cols,
                model_name=model_name,
                n_splits=5
            )

            res["feature_set"] = feature_set_name
            rows_acc_logit.append(res)

        except Exception as e:
            print("Failed:", feature_set_name, model_name, e)

logit_acc_results = pd.DataFrame(rows_acc_logit)

logit_acc_results = logit_acc_results.sort_values(
    ["accuracy", "balanced_accuracy", "auc"],
    ascending=False
)

display(
    logit_acc_results[
        [
            "feature_set",
            "model",
            "n_features",
            "accuracy",
            "balanced_accuracy",
            "f1",
            "auc",
            "up_rate_true",
            "up_rate_pred",
            "avg_threshold"
        ]
    ].head(30)
)

Running | Core Text current | Logit_L1_C0.001 | n_features=35
Running | Core Text current | Logit_L2_C0.001 | n_features=35
Running | Core Text current | Logit_L1_C0.005 | n_features=35
Running | Core Text current | Logit_L2_C0.005 | n_features=35
Running | Core Text current | Logit_L1_C0.01 | n_features=35
Running | Core Text current | Logit_L2_C0.01 | n_features=35
Running | Core Text current | Logit_L1_C0.03 | n_features=35
Running | Core Text current | Logit_L2_C0.03 | n_features=35
Running | Core Text current | Logit_L1_C0.05 | n_features=35
Running | Core Text current | Logit_L2_C0.05 | n_features=35
Running | Core Text current | Logit_L1_C0.1 | n_features=35
Running | Core Text current | Logit_L2_C0.1 | n_features=35
Running | Core Text current | Logit_L1_C0.3 | n_features=35
Running | Core Text current | Logit_L2_C0.3 | n_features=35
Running | Core Text current | Logit_L1_C0.5 | n_features=35
Running | Core Text current | Logit_L2_C0.5 | n_features=35
Running | Core Text curren

,feature_set,model,n_features,accuracy,balanced_accuracy,f1,auc,up_rate_true,up_rate_pred,avg_threshold
13,Core Text current,Logit_L1_C0.3,35,0.595918,0.539138,0.713043,0.550138,0.591837,0.816327,0.490
6,Core Text current,Logit_L2_C0.01,35,0.595918,0.536034,0.716332,0.530759,0.591837,0.832653,0.538
12,Core Text current,Logit_L2_C0.1,35,0.595918,0.531379,0.721127,0.516621,0.591837,0.857143,0.518
24,Core Text current+lag,Logit_L2_C0.001,146,0.591837,0.541897,0.702381,0.499586,0.591837,0.779592,0.534
1,Core Text current,Logit_L1_C0.001,35,0.591837,0.500000,0.743590,0.500000,0.591837,1.000000,0.100
3,Core Text current,Logit_L1_C0.005,35,0.591837,0.500000,0.743590,0.500000,0.591837,1.000000,0.100
5,Core Text current,Logit_L1_C0.01,35,0.591837,0.500000,0.743590,0.500000,0.591837,1.000000,0.100
7,Core Text current,Logit_L1_C0.03,35,0.591837,0.500000,0.743590,0.500000,0.591837,1.000000,0.100
23,Core Text current+lag,Logit_L1_C0.001,146,0.591837,0.500000,0.743590,0.500000,0.591837,1.000000,0.100
25,Core Text current+lag,Logit_L1_C0.005,146,0.591837,0.500000,0.743590,0.500000,0.591837,1.000000,0.100


In [38]:
logit_acc_results.to_csv(
    DATA_DIR / "logistic_accuracy_optimized_core_topic_4h.csv",
    index=False
)

print("Saved:", DATA_DIR / "logistic_accuracy_optimized_core_topic_4h.csv")

Saved: /content/drive/MyDrive/bigquery/logistic_accuracy_optimized_core_topic_4h.csv


In [39]:
credible_logit_results = logit_acc_results[
    (logit_acc_results["up_rate_pred"] >= 0.45) &
    (logit_acc_results["up_rate_pred"] <= 0.85)
].copy()

credible_logit_results = credible_logit_results.sort_values(
    ["accuracy", "balanced_accuracy", "auc"],
    ascending=False
)

display(
    credible_logit_results[
        [
            "feature_set",
            "model",
            "n_features",
            "accuracy",
            "balanced_accuracy",
            "f1",
            "auc",
            "up_rate_true",
            "up_rate_pred",
            "avg_threshold"
        ]
    ].head(20)
)

,feature_set,model,n_features,accuracy,balanced_accuracy,f1,auc,up_rate_true,up_rate_pred,avg_threshold
13,Core Text current,Logit_L1_C0.3,35,0.595918,0.539138,0.713043,0.550138,0.591837,0.816327,0.490
6,Core Text current,Logit_L2_C0.01,35,0.595918,0.536034,0.716332,0.530759,0.591837,0.832653,0.538
24,Core Text current+lag,Logit_L2_C0.001,146,0.591837,0.541897,0.702381,0.499586,0.591837,0.779592,0.534
15,Core Text current,Logit_L1_C0.5,35,0.587755,0.541552,0.694864,0.547655,0.591837,0.759184,0.508
2,Core Text current,Logit_L2_C0.001,35,0.587755,0.535345,0.702065,0.500138,0.591837,0.791837,0.548
14,Core Text current,Logit_L2_C0.3,35,0.587755,0.527586,0.710602,0.498000,0.591837,0.832653,0.504
26,Core Text current+lag,Logit_L2_C0.005,146,0.575510,0.523448,0.692308,0.502138,0.591837,0.787755,0.526
4,Core Text current,Logit_L2_C0.005,35,0.571429,0.518448,0.690265,0.525793,0.591837,0.791837,0.540
35,Core Text current+lag,Logit_L1_C0.3,146,0.571429,0.516897,0.692082,0.500069,0.591837,0.800000,0.498
92,Financial + Core Text current+lag,Logit_L2_C0.005,183,0.571429,0.507586,0.702550,0.475448,0.591837,0.848980,0.518


In [40]:
import numpy as np
import pandas as pd

# 复制原始 4h 数据
df4_meaningful = df4_core.copy()

# 去掉未来 4h return 绝对值最小的 25%
# 你也可以改成 0.20 / 0.30 / 0.40 做 sensitivity analysis
RETURN_FILTER_Q = 0.25

return_threshold = df4_meaningful["future_ret_next"].abs().quantile(RETURN_FILTER_Q)

df4_meaningful = df4_meaningful[
    df4_meaningful["future_ret_next"].abs() >= return_threshold
].copy()

# 重新定义方向标签
df4_meaningful["target_direction"] = (
    df4_meaningful["future_ret_next"] > 0
).astype(int)

print("Original shape:", df4_core.shape)
print("Filtered shape:", df4_meaningful.shape)
print("Removed rows:", len(df4_core) - len(df4_meaningful))
print("Return threshold:", return_threshold)

print("\nOriginal target distribution:")
print(df4_core["target_direction"].value_counts(normalize=True))

print("\nFiltered target distribution:")
print(df4_meaningful["target_direction"].value_counts(normalize=True))

Original shape: (297, 201)
Filtered shape: (223, 201)
Removed rows: 74
Return threshold: 0.004142015839454061

Original target distribution:
target_direction
1    0.575758
0    0.424242
Name: proportion, dtype: float64

Filtered target distribution:
target_direction
1    0.600897
0    0.399103
Name: proportion, dtype: float64


In [41]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

from xgboost import XGBClassifier


def make_model_for_meaningful(model_name):
    if model_name == "Baseline":
        return DummyClassifier(strategy="most_frequent")

    if model_name.startswith("Logit_L1"):
        C = float(model_name.split("_C")[-1])
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                penalty="l1",
                C=C,
                solver="liblinear",
                max_iter=5000,
                class_weight=None
            ))
        ])

    if model_name.startswith("Logit_L2"):
        C = float(model_name.split("_C")[-1])
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                penalty="l2",
                C=C,
                solver="lbfgs",
                max_iter=5000,
                class_weight=None
            ))
        ])

    if model_name.startswith("XGB"):
        parts = model_name.split("_")
        depth = int(parts[1].replace("d", ""))
        lr = float(parts[2].replace("lr", ""))
        n_est = int(parts[3].replace("n", ""))

        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", XGBClassifier(
                n_estimators=n_est,
                max_depth=depth,
                learning_rate=lr,
                subsample=0.8,
                colsample_bytree=0.8,
                eval_metric="logloss",
                random_state=42,
                n_jobs=-1
            ))
        ])

    raise ValueError(model_name)


def find_best_threshold(y_true, prob, metric="accuracy"):
    thresholds = np.linspace(0.20, 0.80, 61)
    best_t = 0.50
    best_score = -999

    for t in thresholds:
        pred = (prob >= t).astype(int)

        if metric == "accuracy":
            score = accuracy_score(y_true, pred)
        elif metric == "balanced_accuracy":
            score = balanced_accuracy_score(y_true, pred)
        else:
            raise ValueError("metric must be accuracy or balanced_accuracy")

        if score > best_score:
            best_score = score
            best_t = t

    return best_t


def evaluate_direction_model(df, feature_cols, model_name, threshold_metric="accuracy", n_splits=5):
    data = df.sort_values("timestamp_utc").reset_index(drop=True).copy()

    X = data[feature_cols].replace([np.inf, -np.inf], np.nan)
    y = data["target_direction"].astype(int).values

    tscv = TimeSeriesSplit(n_splits=n_splits)

    y_true_all = []
    y_pred_all = []
    y_prob_all = []
    thresholds = []

    for train_idx, test_idx in tscv.split(X):
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]

        y_train = y[train_idx]
        y_test = y[test_idx]

        if len(np.unique(y_train)) < 2:
            continue

        model = make_model_for_meaningful(model_name)
        model.fit(X_train, y_train)

        if model_name == "Baseline":
            pred = model.predict(X_test)
            prob = pred
            threshold = np.nan
        else:
            train_prob = model.predict_proba(X_train)[:, 1]
            test_prob = model.predict_proba(X_test)[:, 1]

            threshold = find_best_threshold(
                y_train,
                train_prob,
                metric=threshold_metric
            )

            pred = (test_prob >= threshold).astype(int)
            prob = test_prob

        y_true_all.extend(y_test)
        y_pred_all.extend(pred)
        y_prob_all.extend(prob)
        thresholds.append(threshold)

    y_true_all = np.array(y_true_all)
    y_pred_all = np.array(y_pred_all)
    y_prob_all = np.array(y_prob_all)

    try:
        auc = roc_auc_score(y_true_all, y_prob_all)
    except:
        auc = np.nan

    return {
        "model": model_name,
        "n_features": len(feature_cols),
        "accuracy": accuracy_score(y_true_all, y_pred_all),
        "balanced_accuracy": balanced_accuracy_score(y_true_all, y_pred_all),
        "f1": f1_score(y_true_all, y_pred_all, zero_division=0),
        "auc": auc,
        "up_rate_true": y_true_all.mean(),
        "up_rate_pred": y_pred_all.mean(),
        "avg_threshold": np.nanmean(thresholds),
        "n_eval": len(y_true_all),
        "confusion_matrix": confusion_matrix(y_true_all, y_pred_all).tolist()
    }

In [42]:
feature_sets_meaningful = {
    "Baseline": [],
    "Core Text current": core_text_cols,
    "Core Text current+lag": core_text_cols + lag_cols,
}

# 如果你也想对比金融变量，取消下面这段
feature_sets_meaningful.update({
    "Financial-only": financial_cols,
    "Financial + Core Text current": financial_cols + core_text_cols,
    "Financial + Core Text current+lag": financial_cols + core_text_cols + lag_cols,
})

models_meaningful = ["Baseline"]

for C in [0.01, 0.03, 0.05, 0.1, 0.3, 0.5, 1.0, 2.0]:
    models_meaningful.append(f"Logit_L1_C{C}")
    models_meaningful.append(f"Logit_L2_C{C}")

for d in [1, 2]:
    for lr in [0.01, 0.03, 0.05]:
        for n in [50, 100, 200]:
            models_meaningful.append(f"XGB_d{d}_lr{lr}_n{n}")

rows_meaningful = []

for feature_set_name, cols in feature_sets_meaningful.items():
    if feature_set_name == "Baseline":
        run_models = ["Baseline"]
    else:
        run_models = [m for m in models_meaningful if m != "Baseline"]

    for model_name in run_models:
        print(f"Running meaningful moves | {feature_set_name} | {model_name}")

        try:
            res = evaluate_direction_model(
                df4_meaningful,
                feature_cols=cols,
                model_name=model_name,
                threshold_metric="accuracy",
                n_splits=5
            )

            res["feature_set"] = feature_set_name
            res["return_filter_q"] = RETURN_FILTER_Q
            res["return_threshold"] = return_threshold
            rows_meaningful.append(res)

        except Exception as e:
            print("Failed:", feature_set_name, model_name, e)

meaningful_results = pd.DataFrame(rows_meaningful)

meaningful_results = meaningful_results.sort_values(
    ["accuracy", "balanced_accuracy", "auc"],
    ascending=False
)

display(
    meaningful_results[
        [
            "feature_set",
            "model",
            "n_features",
            "accuracy",
            "balanced_accuracy",
            "f1",
            "auc",
            "up_rate_true",
            "up_rate_pred",
            "avg_threshold",
            "return_filter_q"
        ]
    ].head(30)
)

meaningful_results.to_csv(
    DATA_DIR / "meaningful_moves_core_topic_4h_results.csv",
    index=False
)

print("Saved:", DATA_DIR / "meaningful_moves_core_topic_4h_results.csv")

Running meaningful moves | Baseline | Baseline
Running meaningful moves | Core Text current | Logit_L1_C0.01
Running meaningful moves | Core Text current | Logit_L2_C0.01
Running meaningful moves | Core Text current | Logit_L1_C0.03
Running meaningful moves | Core Text current | Logit_L2_C0.03
Running meaningful moves | Core Text current | Logit_L1_C0.05
Running meaningful moves | Core Text current | Logit_L2_C0.05
Running meaningful moves | Core Text current | Logit_L1_C0.1
Running meaningful moves | Core Text current | Logit_L2_C0.1
Running meaningful moves | Core Text current | Logit_L1_C0.3
Running meaningful moves | Core Text current | Logit_L2_C0.3
Running meaningful moves | Core Text current | Logit_L1_C0.5
Running meaningful moves | Core Text current | Logit_L2_C0.5
Running meaningful moves | Core Text current | Logit_L1_C1.0
Running meaningful moves | Core Text current | Logit_L2_C1.0
Running meaningful moves | Core Text current | Logit_L1_C2.0
Running meaningful moves | Core 

,feature_set,model,n_features,accuracy,balanced_accuracy,f1,auc,up_rate_true,up_rate_pred,avg_threshold,return_filter_q
2,Core Text current,Logit_L2_C0.01,35,0.648649,0.576577,0.761905,0.607621,0.6,0.875676,0.574,0.25
6,Core Text current,Logit_L2_C0.05,35,0.643243,0.576577,0.753731,0.629535,0.6,0.848649,0.564,0.25
8,Core Text current,Logit_L2_C0.1,35,0.637838,0.569820,0.750929,0.627831,0.6,0.854054,0.544,0.25
34,Core Text current,XGB_d2_lr0.05_n200,35,0.632432,0.583333,0.730159,0.554906,0.6,0.762162,0.478,0.25
9,Core Text current,Logit_L1_C0.3,35,0.632432,0.574324,0.738462,0.602447,0.6,0.805405,0.482,0.25
15,Core Text current,Logit_L1_C2.0,35,0.632432,0.567568,0.744361,0.591429,0.6,0.837838,0.462,0.25
11,Core Text current,Logit_L1_C0.5,35,0.627027,0.560811,0.741573,0.606647,0.6,0.843243,0.456,0.25
13,Core Text current,Logit_L1_C1.0,35,0.627027,0.558559,0.743494,0.617969,0.6,0.854054,0.448,0.25
4,Core Text current,Logit_L2_C0.03,35,0.627027,0.551802,0.749091,0.624787,0.6,0.886486,0.546,0.25
10,Core Text current,Logit_L2_C0.3,35,0.621622,0.556306,0.736842,0.617361,0.6,0.837838,0.496,0.25


Saved: /content/drive/MyDrive/bigquery/meaningful_moves_core_topic_4h_results.csv


In [43]:
filter_q_list = [0.10, 0.20, 0.25, 0.30, 0.40]

sensitivity_rows = []

best_models_to_try = [
    "Logit_L1_C0.05",
    "Logit_L1_C0.1",
    "Logit_L1_C0.5",
    "Logit_L2_C0.05",
    "Logit_L2_C0.1",
]

for q in filter_q_list:
    temp_df = df4_core.copy()
    temp_threshold = temp_df["future_ret_next"].abs().quantile(q)

    temp_df = temp_df[
        temp_df["future_ret_next"].abs() >= temp_threshold
    ].copy()

    temp_df["target_direction"] = (temp_df["future_ret_next"] > 0).astype(int)

    print("\n" + "=" * 80)
    print("Filter q:", q)
    print("Shape:", temp_df.shape)
    print("Target distribution:")
    print(temp_df["target_direction"].value_counts(normalize=True))

    for model_name in best_models_to_try:
        for feature_set_name, cols in {
            "Core Text current": core_text_cols,
            "Core Text current+lag": core_text_cols + lag_cols,
        }.items():

            try:
                res = evaluate_direction_model(
                    temp_df,
                    feature_cols=cols,
                    model_name=model_name,
                    threshold_metric="accuracy",
                    n_splits=5
                )

                res["filter_q"] = q
                res["return_threshold"] = temp_threshold
                res["feature_set"] = feature_set_name
                sensitivity_rows.append(res)

            except Exception as e:
                print("Failed:", q, feature_set_name, model_name, e)

sensitivity_results = pd.DataFrame(sensitivity_rows)

sensitivity_results = sensitivity_results.sort_values(
    ["accuracy", "balanced_accuracy", "auc"],
    ascending=False
)

display(
    sensitivity_results[
        [
            "filter_q",
            "feature_set",
            "model",
            "accuracy",
            "balanced_accuracy",
            "auc",
            "up_rate_true",
            "up_rate_pred",
            "return_threshold"
        ]
    ].head(30)
)

sensitivity_results.to_csv(
    DATA_DIR / "meaningful_moves_sensitivity_4h.csv",
    index=False
)


Filter q: 0.1
Shape: (267, 201)
Target distribution:
target_direction
1    0.588015
0    0.411985
Name: proportion, dtype: float64

Filter q: 0.2
Shape: (237, 201)
Target distribution:
target_direction
1    0.607595
0    0.392405
Name: proportion, dtype: float64

Filter q: 0.25
Shape: (223, 201)
Target distribution:
target_direction
1    0.600897
0    0.399103
Name: proportion, dtype: float64

Filter q: 0.3
Shape: (208, 201)
Target distribution:
target_direction
1    0.600962
0    0.399038
Name: proportion, dtype: float64

Filter q: 0.4
Shape: (178, 201)
Target distribution:
target_direction
1    0.606742
0    0.393258
Name: proportion, dtype: float64


,filter_q,feature_set,model,accuracy,balanced_accuracy,auc,up_rate_true,up_rate_pred,return_threshold
48,0.40,Core Text current,Logit_L2_C0.1,0.655172,0.592305,0.603070,0.606897,0.813793,0.006786
44,0.40,Core Text current,Logit_L1_C0.5,0.648276,0.592803,0.601874,0.606897,0.779310,0.006786
46,0.40,Core Text current,Logit_L2_C0.05,0.648276,0.586623,0.605064,0.606897,0.806897,0.006786
18,0.20,Core Text current,Logit_L2_C0.1,0.646154,0.560973,0.616113,0.605128,0.917949,0.002784
16,0.20,Core Text current,Logit_L2_C0.05,0.646154,0.560973,0.613361,0.605128,0.917949,0.002784
26,0.25,Core Text current,Logit_L2_C0.05,0.643243,0.576577,0.629535,0.600000,0.848649,0.004142
28,0.25,Core Text current,Logit_L2_C0.1,0.637838,0.569820,0.627831,0.600000,0.854054,0.004142
14,0.20,Core Text current,Logit_L1_C0.5,0.635897,0.566036,0.593000,0.605128,0.846154,0.002784
34,0.30,Core Text current,Logit_L1_C0.5,0.629412,0.575980,0.600634,0.600000,0.782353,0.004975
36,0.30,Core Text current,Logit_L2_C0.05,0.629412,0.571078,0.616061,0.600000,0.805882,0.004975


In [44]:
def evaluate_confidence_based_prediction(
    df,
    feature_cols,
    model_name,
    confidence_threshold=0.60,
    n_splits=5
):
    data = df.sort_values("timestamp_utc").reset_index(drop=True).copy()

    X = data[feature_cols].replace([np.inf, -np.inf], np.nan)
    y = data["target_direction"].astype(int).values

    tscv = TimeSeriesSplit(n_splits=n_splits)

    y_true_all = []
    y_pred_all = []
    y_prob_all = []
    is_confident_all = []

    for train_idx, test_idx in tscv.split(X):
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]

        y_train = y[train_idx]
        y_test = y[test_idx]

        if len(np.unique(y_train)) < 2:
            continue

        model = make_model_for_meaningful(model_name)
        model.fit(X_train, y_train)

        prob = model.predict_proba(X_test)[:, 1]

        # confidence rule:
        # prob >= threshold -> confident up
        # prob <= 1 - threshold -> confident down
        confident_mask = (
            (prob >= confidence_threshold) |
            (prob <= 1 - confidence_threshold)
        )

        pred = (prob >= 0.5).astype(int)

        y_true_all.extend(y_test)
        y_pred_all.extend(pred)
        y_prob_all.extend(prob)
        is_confident_all.extend(confident_mask)

    y_true_all = np.array(y_true_all)
    y_pred_all = np.array(y_pred_all)
    y_prob_all = np.array(y_prob_all)
    is_confident_all = np.array(is_confident_all)

    coverage = is_confident_all.mean()

    if is_confident_all.sum() == 0:
        return {
            "model": model_name,
            "confidence_threshold": confidence_threshold,
            "n_features": len(feature_cols),
            "coverage": 0,
            "confident_accuracy": np.nan,
            "confident_balanced_accuracy": np.nan,
            "confident_f1": np.nan,
            "overall_accuracy": accuracy_score(y_true_all, y_pred_all),
            "overall_auc": roc_auc_score(y_true_all, y_prob_all),
            "n_confident": 0,
            "n_eval": len(y_true_all),
            "up_rate_true_confident": np.nan,
            "up_rate_pred_confident": np.nan,
        }

    y_true_conf = y_true_all[is_confident_all]
    y_pred_conf = y_pred_all[is_confident_all]

    try:
        overall_auc = roc_auc_score(y_true_all, y_prob_all)
    except:
        overall_auc = np.nan

    return {
        "model": model_name,
        "confidence_threshold": confidence_threshold,
        "n_features": len(feature_cols),
        "coverage": coverage,
        "confident_accuracy": accuracy_score(y_true_conf, y_pred_conf),
        "confident_balanced_accuracy": balanced_accuracy_score(y_true_conf, y_pred_conf),
        "confident_f1": f1_score(y_true_conf, y_pred_conf, zero_division=0),
        "overall_accuracy": accuracy_score(y_true_all, y_pred_all),
        "overall_auc": overall_auc,
        "n_confident": int(is_confident_all.sum()),
        "n_eval": len(y_true_all),
        "up_rate_true_confident": y_true_conf.mean(),
        "up_rate_pred_confident": y_pred_conf.mean(),
    }

In [45]:
confidence_models = [
    "Logit_L1_C0.05",
    "Logit_L1_C0.1",
    "Logit_L1_C0.5",
    "Logit_L2_C0.05",
    "Logit_L2_C0.1",
    "Logit_L2_C0.5",
]

confidence_feature_sets = {
    "Core Text current": core_text_cols,
    "Core Text current+lag": core_text_cols + lag_cols,
}

confidence_thresholds = [0.55, 0.60, 0.65, 0.70, 0.75]

conf_rows = []

for feature_set_name, cols in confidence_feature_sets.items():
    for model_name in confidence_models:
        for conf_t in confidence_thresholds:

            print(f"Running confidence | {feature_set_name} | {model_name} | threshold={conf_t}")

            try:
                res = evaluate_confidence_based_prediction(
                    df4_core,
                    feature_cols=cols,
                    model_name=model_name,
                    confidence_threshold=conf_t,
                    n_splits=5
                )

                res["feature_set"] = feature_set_name
                conf_rows.append(res)

            except Exception as e:
                print("Failed:", feature_set_name, model_name, conf_t, e)

confidence_results = pd.DataFrame(conf_rows)

confidence_results = confidence_results.sort_values(
    ["confident_accuracy", "coverage"],
    ascending=False
)

display(
    confidence_results[
        [
            "feature_set",
            "model",
            "confidence_threshold",
            "coverage",
            "confident_accuracy",
            "confident_balanced_accuracy",
            "overall_accuracy",
            "overall_auc",
            "n_confident",
            "n_eval",
            "up_rate_true_confident",
            "up_rate_pred_confident"
        ]
    ].head(30)
)

confidence_results.to_csv(
    DATA_DIR / "confidence_based_core_topic_4h_results.csv",
    index=False
)

print("Saved:", DATA_DIR / "confidence_based_core_topic_4h_results.csv")

Running confidence | Core Text current | Logit_L1_C0.05 | threshold=0.55
Running confidence | Core Text current | Logit_L1_C0.05 | threshold=0.6
Running confidence | Core Text current | Logit_L1_C0.05 | threshold=0.65
Running confidence | Core Text current | Logit_L1_C0.05 | threshold=0.7
Running confidence | Core Text current | Logit_L1_C0.05 | threshold=0.75
Running confidence | Core Text current | Logit_L1_C0.1 | threshold=0.55
Running confidence | Core Text current | Logit_L1_C0.1 | threshold=0.6
Running confidence | Core Text current | Logit_L1_C0.1 | threshold=0.65
Running confidence | Core Text current | Logit_L1_C0.1 | threshold=0.7
Running confidence | Core Text current | Logit_L1_C0.1 | threshold=0.75
Running confidence | Core Text current | Logit_L1_C0.5 | threshold=0.55
Running confidence | Core Text current | Logit_L1_C0.5 | threshold=0.6
Running confidence | Core Text current | Logit_L1_C0.5 | threshold=0.65
Running confidence | Core Text current | Logit_L1_C0.5 | thresho

,feature_set,model,confidence_threshold,coverage,confident_accuracy,confident_balanced_accuracy,overall_accuracy,overall_auc,n_confident,n_eval,up_rate_true_confident,up_rate_pred_confident
38,Core Text current+lag,Logit_L1_C0.1,0.70,0.012245,1.000000,1.000000,0.571429,0.495966,3,245,1.000000,1.000000
8,Core Text current,Logit_L1_C0.1,0.70,0.004082,1.000000,1.000000,0.579592,0.506448,1,245,1.000000,1.000000
37,Core Text current+lag,Logit_L1_C0.1,0.65,0.048980,0.666667,0.500000,0.571429,0.495897,12,245,0.666667,1.000000
7,Core Text current,Logit_L1_C0.1,0.65,0.036735,0.666667,0.500000,0.579592,0.506448,9,245,0.666667,1.000000
13,Core Text current,Logit_L1_C0.5,0.70,0.359184,0.636364,0.572650,0.587755,0.547586,88,245,0.590909,0.863636
14,Core Text current,Logit_L1_C0.5,0.75,0.244898,0.633333,0.565714,0.587755,0.547655,60,245,0.583333,0.916667
18,Core Text current,Logit_L2_C0.05,0.70,0.489796,0.625000,0.531250,0.591837,0.520759,120,245,0.600000,0.975000
11,Core Text current,Logit_L1_C0.5,0.60,0.636735,0.621795,0.551647,0.587755,0.547655,156,245,0.602564,0.852564
16,Core Text current,Logit_L2_C0.05,0.60,0.759184,0.618280,0.528246,0.591837,0.520759,186,245,0.607527,0.924731
12,Core Text current,Logit_L1_C0.5,0.65,0.493878,0.611570,0.550704,0.587755,0.547586,121,245,0.586777,0.859504


Saved: /content/drive/MyDrive/bigquery/confidence_based_core_topic_4h_results.csv


In [46]:
credible_confidence_results = confidence_results[
    confidence_results["coverage"] >= 0.20
].copy()

credible_confidence_results = credible_confidence_results.sort_values(
    ["confident_accuracy", "confident_balanced_accuracy", "coverage"],
    ascending=False
)

display(
    credible_confidence_results[
        [
            "feature_set",
            "model",
            "confidence_threshold",
            "coverage",
            "confident_accuracy",
            "confident_balanced_accuracy",
            "overall_accuracy",
            "overall_auc",
            "n_confident",
            "up_rate_true_confident",
            "up_rate_pred_confident"
        ]
    ].head(20)
)

,feature_set,model,confidence_threshold,coverage,confident_accuracy,confident_balanced_accuracy,overall_accuracy,overall_auc,n_confident,up_rate_true_confident,up_rate_pred_confident
13,Core Text current,Logit_L1_C0.5,0.70,0.359184,0.636364,0.572650,0.587755,0.547586,88,0.590909,0.863636
14,Core Text current,Logit_L1_C0.5,0.75,0.244898,0.633333,0.565714,0.587755,0.547655,60,0.583333,0.916667
18,Core Text current,Logit_L2_C0.05,0.70,0.489796,0.625000,0.531250,0.591837,0.520759,120,0.600000,0.975000
11,Core Text current,Logit_L1_C0.5,0.60,0.636735,0.621795,0.551647,0.587755,0.547655,156,0.602564,0.852564
16,Core Text current,Logit_L2_C0.05,0.60,0.759184,0.618280,0.528246,0.591837,0.520759,186,0.607527,0.924731
12,Core Text current,Logit_L1_C0.5,0.65,0.493878,0.611570,0.550704,0.587755,0.547586,121,0.586777,0.859504
23,Core Text current,Logit_L2_C0.1,0.70,0.530612,0.607692,0.522436,0.591837,0.516621,130,0.600000,0.930769
17,Core Text current,Logit_L2_C0.05,0.65,0.612245,0.606667,0.524130,0.591837,0.520759,150,0.593333,0.946667
22,Core Text current,Logit_L2_C0.1,0.65,0.640816,0.605096,0.523641,0.591837,0.516621,157,0.598726,0.917197
15,Core Text current,Logit_L2_C0.05,0.55,0.881633,0.601852,0.520716,0.591837,0.520759,216,0.597222,0.921296


In [47]:
import pandas as pd
import numpy as np
from pathlib import Path

# 核心 topic
core_topics = [
    "inventory_eia",
    "diplomacy_ceasefire",
    "military_escalation",
    "hormuz_shipping",
    "sanctions_exports"
]

# 如果没有 DATA_DIR，就补一下
if "DATA_DIR" not in globals():
    DATA_DIR = Path("/content/drive/MyDrive/bigquery")

# 当前 topic 特征：不包括 lag
def get_topic_current_cols(df, topic):
    return [
        c for c in df.columns
        if (
            c.startswith(topic + "_")
            or c.startswith("log1p_" + topic + "_")
        )
        and not c.endswith("_lag1")
        and not c.endswith("_lag2")
        and not c.endswith("_lag3")
        and pd.api.types.is_numeric_dtype(df[c])
    ]

topic_current_cols = {
    topic: get_topic_current_cols(df4_core, topic)
    for topic in core_topics
}

for topic, cols in topic_current_cols.items():
    print(topic, len(cols))
    print(cols[:10])

inventory_eia 7
['inventory_eia_count', 'inventory_eia_source_count', 'inventory_eia_relevance_sum', 'inventory_eia_relevance_mean', 'log1p_inventory_eia_count', 'log1p_inventory_eia_source_count', 'log1p_inventory_eia_relevance_sum']
diplomacy_ceasefire 7
['diplomacy_ceasefire_count', 'diplomacy_ceasefire_source_count', 'diplomacy_ceasefire_relevance_sum', 'diplomacy_ceasefire_relevance_mean', 'log1p_diplomacy_ceasefire_count', 'log1p_diplomacy_ceasefire_source_count', 'log1p_diplomacy_ceasefire_relevance_sum']
military_escalation 7
['military_escalation_count', 'military_escalation_source_count', 'military_escalation_relevance_sum', 'military_escalation_relevance_mean', 'log1p_military_escalation_count', 'log1p_military_escalation_source_count', 'log1p_military_escalation_relevance_sum']
hormuz_shipping 7
['hormuz_shipping_count', 'hormuz_shipping_source_count', 'hormuz_shipping_relevance_sum', 'hormuz_shipping_relevance_mean', 'log1p_hormuz_shipping_count', 'log1p_hormuz_shipping_so

In [48]:
df_topic_sens = df4_core.sort_values("timestamp_utc").reset_index(drop=True).copy()

created_lag_cols = []

for topic in core_topics:
    base_cols = topic_current_cols[topic]

    # 只给 count / source_count / relevance_sum / log1p 类特征加 lag
    lag_base_cols = [
        c for c in base_cols
        if (
            c.endswith("_count")
            or c.endswith("_source_count")
            or c.endswith("_relevance_sum")
            or c.startswith("log1p_")
        )
    ]

    for col in lag_base_cols:
        for lag in [1, 2, 3]:
            lag_col = f"{col}_lag{lag}"
            if lag_col not in df_topic_sens.columns:
                df_topic_sens[lag_col] = df_topic_sens[col].shift(lag)
            created_lag_cols.append(lag_col)

created_lag_cols = list(dict.fromkeys(created_lag_cols))
df_topic_sens[created_lag_cols] = df_topic_sens[created_lag_cols].fillna(0)

def get_topic_lag_cols(df, topic):
    return [
        c for c in df.columns
        if (
            c.startswith(topic + "_")
            or c.startswith("log1p_" + topic + "_")
        )
        and (
            c.endswith("_lag1")
            or c.endswith("_lag2")
            or c.endswith("_lag3")
        )
        and pd.api.types.is_numeric_dtype(df[c])
    ]

topic_lag_cols = {
    topic: get_topic_lag_cols(df_topic_sens, topic)
    for topic in core_topics
}

topic_all_cols = {
    topic: topic_current_cols[topic] + topic_lag_cols[topic]
    for topic in core_topics
}

print("Lag cols created:", len(created_lag_cols))

for topic in core_topics:
    print(topic, "current:", len(topic_current_cols[topic]), "lag:", len(topic_lag_cols[topic]), "total:", len(topic_all_cols[topic]))

Lag cols created: 90
inventory_eia current: 7 lag: 18 total: 25
diplomacy_ceasefire current: 7 lag: 18 total: 25
military_escalation current: 7 lag: 18 total: 25
hormuz_shipping current: 7 lag: 18 total: 25
sanctions_exports current: 7 lag: 18 total: 25


In [49]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

def make_logit_model(model_name):
    if model_name.startswith("Logit_L1"):
        C = float(model_name.split("_C")[-1])
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                penalty="l1",
                solver="liblinear",
                C=C,
                max_iter=5000,
                class_weight=None
            ))
        ])

    if model_name.startswith("Logit_L2"):
        C = float(model_name.split("_C")[-1])
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                penalty="l2",
                solver="lbfgs",
                C=C,
                max_iter=5000,
                class_weight=None
            ))
        ])

    raise ValueError(model_name)


def find_best_threshold_train(y_true, prob, metric="accuracy"):
    thresholds = np.linspace(0.20, 0.80, 61)
    best_t = 0.50
    best_score = -999

    for t in thresholds:
        pred = (prob >= t).astype(int)

        if metric == "accuracy":
            score = accuracy_score(y_true, pred)
        elif metric == "balanced_accuracy":
            score = balanced_accuracy_score(y_true, pred)
        else:
            raise ValueError("metric must be accuracy or balanced_accuracy")

        if score > best_score:
            best_score = score
            best_t = t

    return best_t


def prepare_filtered_df(df, filter_q=0.0):
    temp = df.copy()

    if filter_q > 0:
        threshold = temp["future_ret_next"].abs().quantile(filter_q)
        temp = temp[temp["future_ret_next"].abs() >= threshold].copy()
    else:
        threshold = 0.0

    temp["target_direction"] = (temp["future_ret_next"] > 0).astype(int)

    return temp, threshold


def evaluate_topic_model(
    df,
    feature_cols,
    model_name="Logit_L2_C0.1",
    filter_q=0.0,
    threshold_metric="accuracy",
    n_splits=5
):
    data, return_threshold = prepare_filtered_df(df, filter_q=filter_q)
    data = data.sort_values("timestamp_utc").reset_index(drop=True).copy()

    X = data[feature_cols].replace([np.inf, -np.inf], np.nan)
    y = data["target_direction"].astype(int).values

    tscv = TimeSeriesSplit(n_splits=n_splits)

    y_true_all = []
    y_pred_all = []
    y_prob_all = []
    thresholds = []

    for train_idx, test_idx in tscv.split(X):
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]
        y_train = y[train_idx]
        y_test = y[test_idx]

        if len(np.unique(y_train)) < 2:
            continue

        model = make_logit_model(model_name)
        model.fit(X_train, y_train)

        train_prob = model.predict_proba(X_train)[:, 1]
        test_prob = model.predict_proba(X_test)[:, 1]

        best_t = find_best_threshold_train(
            y_train,
            train_prob,
            metric=threshold_metric
        )

        test_pred = (test_prob >= best_t).astype(int)

        y_true_all.extend(y_test)
        y_pred_all.extend(test_pred)
        y_prob_all.extend(test_prob)
        thresholds.append(best_t)

    y_true_all = np.array(y_true_all)
    y_pred_all = np.array(y_pred_all)
    y_prob_all = np.array(y_prob_all)

    try:
        auc = roc_auc_score(y_true_all, y_prob_all)
    except:
        auc = np.nan

    return {
        "model": model_name,
        "filter_q": filter_q,
        "return_threshold": return_threshold,
        "n_features": len(feature_cols),
        "n_eval": len(y_true_all),
        "accuracy": accuracy_score(y_true_all, y_pred_all),
        "balanced_accuracy": balanced_accuracy_score(y_true_all, y_pred_all),
        "f1": f1_score(y_true_all, y_pred_all, zero_division=0),
        "auc": auc,
        "up_rate_true": y_true_all.mean(),
        "up_rate_pred": y_pred_all.mean(),
        "avg_threshold": np.mean(thresholds),
        "confusion_matrix": confusion_matrix(y_true_all, y_pred_all).tolist()
    }

In [50]:
filter_q_list = [0.0, 0.20, 0.25, 0.30, 0.40]
models_to_test = ["Logit_L1_C0.5", "Logit_L2_C0.1"]

single_topic_rows = []

for filter_q in filter_q_list:
    for model_name in models_to_test:
        for topic in core_topics:

            # current only
            cols_current = topic_current_cols[topic]

            if len(cols_current) > 0:
                res = evaluate_topic_model(
                    df_topic_sens,
                    feature_cols=cols_current,
                    model_name=model_name,
                    filter_q=filter_q,
                    threshold_metric="accuracy",
                    n_splits=5
                )

                res["analysis"] = "single_topic_current"
                res["topic"] = topic
                res["feature_version"] = "current"
                single_topic_rows.append(res)

            # current + lag
            cols_all = topic_all_cols[topic]

            if len(cols_all) > 0:
                res = evaluate_topic_model(
                    df_topic_sens,
                    feature_cols=cols_all,
                    model_name=model_name,
                    filter_q=filter_q,
                    threshold_metric="accuracy",
                    n_splits=5
                )

                res["analysis"] = "single_topic_current_lag"
                res["topic"] = topic
                res["feature_version"] = "current+lag"
                single_topic_rows.append(res)

single_topic_results = pd.DataFrame(single_topic_rows)

single_topic_results = single_topic_results.sort_values(
    ["filter_q", "accuracy", "balanced_accuracy", "auc"],
    ascending=[True, False, False, False]
)

display(
    single_topic_results[
        [
            "analysis",
            "filter_q",
            "topic",
            "feature_version",
            "model",
            "n_features",
            "accuracy",
            "balanced_accuracy",
            "auc",
            "up_rate_true",
            "up_rate_pred",
            "return_threshold"
        ]
    ].head(50)
)

single_topic_results.to_csv(
    DATA_DIR / "topic_sensitivity_single_topic_4h.csv",
    index=False
)

print("Saved:", DATA_DIR / "topic_sensitivity_single_topic_4h.csv")

,analysis,filter_q,topic,feature_version,model,n_features,accuracy,balanced_accuracy,auc,up_rate_true,up_rate_pred,return_threshold
11,single_topic_current_lag,0.00,inventory_eia,current+lag,Logit_L2_C0.1,25,0.595918,0.517414,0.491517,0.591837,0.930612,0.000000
0,single_topic_current,0.00,inventory_eia,current,Logit_L1_C0.5,7,0.587755,0.501207,0.515966,0.591837,0.971429,0.000000
1,single_topic_current_lag,0.00,inventory_eia,current+lag,Logit_L1_C0.5,25,0.583673,0.508621,0.510483,0.591837,0.910204,0.000000
10,single_topic_current,0.00,inventory_eia,current,Logit_L2_C0.1,7,0.583673,0.497759,0.506310,0.591837,0.967347,0.000000
8,single_topic_current,0.00,sanctions_exports,current,Logit_L1_C0.5,7,0.575510,0.518793,0.511414,0.591837,0.812245,0.000000
9,single_topic_current_lag,0.00,sanctions_exports,current+lag,Logit_L1_C0.5,25,0.575510,0.518793,0.474207,0.591837,0.812245,0.000000
6,single_topic_current,0.00,hormuz_shipping,current,Logit_L1_C0.5,7,0.575510,0.489310,0.381310,0.591837,0.967347,0.000000
19,single_topic_current_lag,0.00,sanctions_exports,current+lag,Logit_L2_C0.1,25,0.563265,0.522414,0.477793,0.591837,0.726531,0.000000
17,single_topic_current_lag,0.00,hormuz_shipping,current+lag,Logit_L2_C0.1,25,0.563265,0.486724,0.436345,0.591837,0.914286,0.000000
7,single_topic_current_lag,0.00,hormuz_shipping,current+lag,Logit_L1_C0.5,25,0.563265,0.483621,0.438552,0.591837,0.930612,0.000000


Saved: /content/drive/MyDrive/bigquery/topic_sensitivity_single_topic_4h.csv


In [51]:
def combine_topics(topics, use_lag=False):
    cols = []

    for topic in topics:
        if use_lag:
            cols.extend(topic_all_cols[topic])
        else:
            cols.extend(topic_current_cols[topic])

    # 去重
    cols = list(dict.fromkeys(cols))
    return cols


leave_one_rows = []

for filter_q in filter_q_list:
    for model_name in models_to_test:
        for use_lag in [False, True]:

            feature_version = "current+lag" if use_lag else "current"

            # all core baseline
            all_core_cols = combine_topics(core_topics, use_lag=use_lag)

            base_res = evaluate_topic_model(
                df_topic_sens,
                feature_cols=all_core_cols,
                model_name=model_name,
                filter_q=filter_q,
                threshold_metric="accuracy",
                n_splits=5
            )

            base_res["analysis"] = "all_core"
            base_res["removed_topic"] = "none"
            base_res["remaining_topics"] = ",".join(core_topics)
            base_res["feature_version"] = feature_version
            leave_one_rows.append(base_res)

            # leave one out
            for removed_topic in core_topics:
                remaining_topics = [t for t in core_topics if t != removed_topic]
                cols = combine_topics(remaining_topics, use_lag=use_lag)

                res = evaluate_topic_model(
                    df_topic_sens,
                    feature_cols=cols,
                    model_name=model_name,
                    filter_q=filter_q,
                    threshold_metric="accuracy",
                    n_splits=5
                )

                res["analysis"] = "leave_one_out"
                res["removed_topic"] = removed_topic
                res["remaining_topics"] = ",".join(remaining_topics)
                res["feature_version"] = feature_version
                leave_one_rows.append(res)

leave_one_results = pd.DataFrame(leave_one_rows)

display(
    leave_one_results[
        [
            "analysis",
            "filter_q",
            "feature_version",
            "model",
            "removed_topic",
            "n_features",
            "accuracy",
            "balanced_accuracy",
            "auc",
            "up_rate_true",
            "up_rate_pred",
            "return_threshold"
        ]
    ].sort_values(
        ["filter_q", "feature_version", "model", "analysis", "accuracy"],
        ascending=[True, True, True, True, False]
    ).head(80)
)

leave_one_results.to_csv(
    DATA_DIR / "topic_sensitivity_leave_one_out_4h.csv",
    index=False
)

print("Saved:", DATA_DIR / "topic_sensitivity_leave_one_out_4h.csv")

,analysis,filter_q,feature_version,model,removed_topic,n_features,accuracy,balanced_accuracy,auc,up_rate_true,up_rate_pred,return_threshold
0,all_core,0.0,current,Logit_L1_C0.5,none,35,0.587755,0.541552,0.547655,0.591837,0.759184,0.000000
3,leave_one_out,0.0,current,Logit_L1_C0.5,military_escalation,28,0.604082,0.552241,0.565586,0.591837,0.791837,0.000000
4,leave_one_out,0.0,current,Logit_L1_C0.5,hormuz_shipping,28,0.600000,0.545690,0.556276,0.591837,0.804082,0.000000
5,leave_one_out,0.0,current,Logit_L1_C0.5,sanctions_exports,28,0.575510,0.534310,0.529448,0.591837,0.730612,0.000000
1,leave_one_out,0.0,current,Logit_L1_C0.5,inventory_eia,28,0.567347,0.504138,0.495586,0.591837,0.844898,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
74,leave_one_out,0.3,current,Logit_L1_C0.5,diplomacy_ceasefire,28,0.594118,0.524510,0.571943,0.600000,0.852941,0.004975
73,leave_one_out,0.3,current,Logit_L1_C0.5,inventory_eia,28,0.558824,0.519608,0.532151,0.600000,0.700000,0.004975
77,leave_one_out,0.3,current,Logit_L1_C0.5,sanctions_exports,28,0.558824,0.512255,0.600346,0.600000,0.735294,0.004975
84,all_core,0.3,current,Logit_L2_C0.1,none,35,0.617647,0.563725,0.613033,0.600000,0.782353,0.004975


Saved: /content/drive/MyDrive/bigquery/topic_sensitivity_leave_one_out_4h.csv


In [52]:
delta_rows = []

group_cols = ["filter_q", "feature_version", "model"]

for keys, group in leave_one_results.groupby(group_cols):
    filter_q, feature_version, model_name = keys

    base = group[group["analysis"] == "all_core"]

    if len(base) == 0:
        continue

    base = base.iloc[0]

    loo = group[group["analysis"] == "leave_one_out"]

    for _, row in loo.iterrows():
        delta_rows.append({
            "filter_q": filter_q,
            "feature_version": feature_version,
            "model": model_name,
            "removed_topic": row["removed_topic"],

            "base_accuracy": base["accuracy"],
            "loo_accuracy": row["accuracy"],
            "delta_accuracy": row["accuracy"] - base["accuracy"],

            "base_balanced_accuracy": base["balanced_accuracy"],
            "loo_balanced_accuracy": row["balanced_accuracy"],
            "delta_balanced_accuracy": row["balanced_accuracy"] - base["balanced_accuracy"],

            "base_auc": base["auc"],
            "loo_auc": row["auc"],
            "delta_auc": row["auc"] - base["auc"],

            "base_up_rate_pred": base["up_rate_pred"],
            "loo_up_rate_pred": row["up_rate_pred"],
        })

leave_one_delta = pd.DataFrame(delta_rows)

# 如果 delta < 0，说明删掉这个 topic 后表现下降，也就是这个 topic 有帮助
display(
    leave_one_delta[
        [
            "filter_q",
            "feature_version",
            "model",
            "removed_topic",
            "delta_accuracy",
            "delta_balanced_accuracy",
            "delta_auc",
            "base_accuracy",
            "loo_accuracy",
            "base_auc",
            "loo_auc"
        ]
    ].sort_values(
        ["filter_q", "delta_auc"],
        ascending=[True, True]
    ).head(80)
)

leave_one_delta.to_csv(
    DATA_DIR / "topic_sensitivity_leave_one_out_delta_4h.csv",
    index=False
)

print("Saved:", DATA_DIR / "topic_sensitivity_leave_one_out_delta_4h.csv")

,filter_q,feature_version,model,removed_topic,delta_accuracy,delta_balanced_accuracy,delta_auc,base_accuracy,loo_accuracy,base_auc,loo_auc
10,0.0,current+lag,Logit_L1_C0.5,inventory_eia,-0.020408,-0.020345,-0.062828,0.563265,0.542857,0.508966,0.446138
0,0.0,current,Logit_L1_C0.5,inventory_eia,-0.020408,-0.037414,-0.052069,0.587755,0.567347,0.547655,0.495586
5,0.0,current,Logit_L2_C0.1,inventory_eia,-0.048980,-0.047586,-0.047931,0.595918,0.546939,0.516621,0.468690
15,0.0,current+lag,Logit_L2_C0.1,inventory_eia,-0.036735,-0.035690,-0.042276,0.559184,0.522449,0.480828,0.438552
6,0.0,current,Logit_L2_C0.1,diplomacy_ceasefire,-0.028571,-0.031897,-0.033862,0.595918,0.567347,0.516621,0.482759
...,...,...,...,...,...,...,...,...,...,...,...
69,0.3,current,Logit_L2_C0.1,sanctions_exports,-0.023529,-0.012255,0.000433,0.617647,0.594118,0.613033,0.613466
68,0.3,current,Logit_L2_C0.1,hormuz_shipping,-0.017647,-0.041667,0.002018,0.617647,0.600000,0.613033,0.615052
74,0.3,current+lag,Logit_L1_C0.5,sanctions_exports,-0.005882,-0.004902,0.003028,0.552941,0.547059,0.546424,0.549452
79,0.3,current+lag,Logit_L2_C0.1,sanctions_exports,-0.017647,-0.012255,0.016292,0.594118,0.576471,0.560698,0.576990


Saved: /content/drive/MyDrive/bigquery/topic_sensitivity_leave_one_out_delta_4h.csv


In [53]:
topic_groups = {
    "all_core": [
        "inventory_eia",
        "diplomacy_ceasefire",
        "military_escalation",
        "hormuz_shipping",
        "sanctions_exports"
    ],

    "geopolitical_risk": [
        "military_escalation",
        "hormuz_shipping",
        "sanctions_exports",
        "diplomacy_ceasefire"
    ],

    "supply_disruption": [
        "hormuz_shipping",
        "sanctions_exports",
        "military_escalation"
    ],

    "deescalation_only": [
        "diplomacy_ceasefire"
    ],

    "inventory_only": [
        "inventory_eia"
    ],

    "conflict_vs_diplomacy": [
        "military_escalation",
        "diplomacy_ceasefire"
    ],

    "inventory_plus_geopolitics": [
        "inventory_eia",
        "military_escalation",
        "hormuz_shipping",
        "sanctions_exports"
    ],
}

combo_rows = []

for filter_q in filter_q_list:
    for model_name in models_to_test:
        for group_name, topics in topic_groups.items():
            for use_lag in [False, True]:

                feature_version = "current+lag" if use_lag else "current"
                cols = combine_topics(topics, use_lag=use_lag)

                if len(cols) == 0:
                    continue

                res = evaluate_topic_model(
                    df_topic_sens,
                    feature_cols=cols,
                    model_name=model_name,
                    filter_q=filter_q,
                    threshold_metric="accuracy",
                    n_splits=5
                )

                res["analysis"] = "topic_group"
                res["topic_group"] = group_name
                res["topics"] = ",".join(topics)
                res["feature_version"] = feature_version

                combo_rows.append(res)

combo_results = pd.DataFrame(combo_rows)

combo_results = combo_results.sort_values(
    ["filter_q", "accuracy", "balanced_accuracy", "auc"],
    ascending=[True, False, False, False]
)

display(
    combo_results[
        [
            "filter_q",
            "topic_group",
            "topics",
            "feature_version",
            "model",
            "n_features",
            "accuracy",
            "balanced_accuracy",
            "auc",
            "up_rate_true",
            "up_rate_pred",
            "return_threshold"
        ]
    ].head(80)
)

combo_results.to_csv(
    DATA_DIR / "topic_sensitivity_topic_groups_4h.csv",
    index=False
)

print("Saved:", DATA_DIR / "topic_sensitivity_topic_groups_4h.csv")

,filter_q,topic_group,topics,feature_version,model,n_features,accuracy,balanced_accuracy,auc,up_rate_true,up_rate_pred,return_threshold
14,0.00,all_core,"inventory_eia,diplomacy_ceasefire,military_esc...",current,Logit_L2_C0.1,35,0.595918,0.531379,0.516621,0.591837,0.857143,0.000000
23,0.00,inventory_only,inventory_eia,current+lag,Logit_L2_C0.1,25,0.595918,0.517414,0.491517,0.591837,0.930612,0.000000
0,0.00,all_core,"inventory_eia,diplomacy_ceasefire,military_esc...",current,Logit_L1_C0.5,35,0.587755,0.541552,0.547655,0.591837,0.759184,0.000000
13,0.00,inventory_plus_geopolitics,"inventory_eia,military_escalation,hormuz_shipp...",current+lag,Logit_L1_C0.5,100,0.587755,0.521379,0.512069,0.591837,0.865306,0.000000
8,0.00,inventory_only,inventory_eia,current,Logit_L1_C0.5,7,0.587755,0.501207,0.515966,0.591837,0.971429,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
72,0.25,geopolitical_risk,"military_escalation,hormuz_shipping,sanctions_...",current,Logit_L2_C0.1,28,0.572973,0.500000,0.518627,0.600000,0.864865,0.004142
60,0.25,supply_disruption,"hormuz_shipping,sanctions_exports,military_esc...",current,Logit_L1_C0.5,21,0.562162,0.481982,0.462503,0.600000,0.897297,0.004142
73,0.25,geopolitical_risk,"military_escalation,hormuz_shipping,sanctions_...",current+lag,Logit_L2_C0.1,100,0.551351,0.500000,0.525444,0.600000,0.756757,0.004142
77,0.25,deescalation_only,diplomacy_ceasefire,current+lag,Logit_L2_C0.1,25,0.545946,0.513514,0.519235,0.600000,0.664865,0.004142


Saved: /content/drive/MyDrive/bigquery/topic_sensitivity_topic_groups_4h.csv
